<a href="https://colab.research.google.com/github/BrenoBragaGalvao/mestrado-arvores-urbanas/blob/main/Segmentacao_area_completa_v2_experimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODO PROGRESSIVO 4 GiB — rota independente com checkpoints (Area1)

**Comece aqui para uma execução em lotes. NÃO clique em "Executar tudo".** As 116 células históricas que aparecem depois deste bloco ainda fazem processamento de rasters inteiros e podem estourar a memória. **Execute somente as células desta seção, na ordem**, repetindo a célula de cada lote até aparecer "CONCLUÍDO". Não é necessário executar as células históricas para usar esta rota.

1. **Instalação + Drive**: execute a primeira célula.
2. **Configuração**: execute a célula que define as funções. Os dados originais nunca são modificados.
3. **Pré-processamento em lotes**: execute repetidamente a célula "ETAPA A". Cada rodada processa no máximo \`TILES_POR_RODADA=30\` quadrantes de 256×256 pixels e salva arquivos \`.npz\` no Google Drive. Pode interromper e continuar em outra sessão repetindo os passos 1–3.
4. **Experimentos E1–E5**: execute a "ETAPA B" *após terminar todos os quadrantes*. Ela processa rótulos existentes, compara classificadores com blocos espaciais, faz ablação e testa K-Means. Reexecuções substituem somente saídas da pasta experimental.
5. **Proximidade em lotes**: execute repetidamente "ETAPA C". Utiliza a classificação treinada na etapa B, poligoniza somente segmentos candidatos próximos de postes, e salva checkpoints por quadrante.
6. **Consolidação E6–E8**: execute a "ETAPA D" ao concluir C. Salva tabelas e relatório em \`/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs/v2_progressivo_256/\`.

**Atenção metodológica:** este é um experimento **novo de cobertura da área em quadrantes independentes**, NÃO é a execução integral equivalente ao Quickshift aplicado ao raster inteiro. Limites dos quadrantes podem cortar copas; a classificação experimental considera todos os rótulos válidos (não usa seleção histórica fixa \`cluster 3\`). A exclusão espacial de 25 m nesta rota usa **distância entre centroides** (mais simples que a separação geométrica por polígonos do E2 original). Os resultados não são diretamente comparáveis à acurácia histórica 0,87. Sem cabos e alturas reais, não é possível validar risco elétrico ou acurácia de altura.

**Preservação:** notebook original, README.md e arquivos Area1 não são modificados. A rota escreve apenas na pasta \`outputs/v2_progressivo_256\`. Ajuste \`TILES_POR_RODADA\` para 10 se houver lentidão ou memória insuficiente.

In [ ]:
# CONFIGURAÇÃO (executar novamente ao reiniciar a sessão)
%pip -q install rasterio geopandas shapely scikit-image scikit-learn joblib
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import defaultdict
import os, json, gc, math, time, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window, transform as window_transform
from rasterio.features import shapes, geometry_mask
from skimage.segmentation import quickshift
from skimage.feature import canny
from skimage.color import rgb2hsv
from shapely.geometry import shape, box, mapping
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, silhouette_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
import itertools, joblib
BASE_PROG = Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno')
AREA_PROG = BASE_PROG / 'Area1'
SAIDA_PROG = BASE_PROG / 'outputs' / 'v2_progressivo_256'
TILE_PROG = 256
TILES_POR_RODADA = 30  # Reduza para 10, se necessário.
DIR_TILES = SAIDA_PROG / 'tiles'
DIR_SPATIAL = SAIDA_PROG / 'spatial_tiles'
for pasta in [SAIDA_PROG, DIR_TILES, DIR_SPATIAL]:
    pasta.mkdir(parents=True, exist_ok=True)
for arquivo in ['AOI_RGB_NEW.tif','AOI_NEW_MULTI.tif','AOI_new.shp',
                'Arvores.gpkg','Grama.gpkg','postes.gpkg']:
    if not (AREA_PROG / arquivo).is_file():
        raise FileNotFoundError(f'Arquivo original ausente no Drive: {arquivo}')
with rasterio.open(AREA_PROG/'AOI_RGB_NEW.tif') as r:
    SHAPE_PROG = (r.height, r.width)
    TRANSFORM_PROG = r.transform
    CRS_PROG = r.crs
    RES_PROG = abs(r.res[0] * r.res[1])
with rasterio.open(AREA_PROG/'AOI_NEW_MULTI.tif') as r:
    if (r.height,r.width) != SHAPE_PROG or r.crs != CRS_PROG:
        raise ValueError('Rasters RGB e multiespectral sem dimensões/CRS compatíveis.')
    if not r.transform.almost_equals(TRANSFORM_PROG):
        warnings.warn('As duas grades raster apresentam deslocamento de origem; extração por janela é aproximação espacial e deve ser descrita.')
AOI_PROG = gpd.read_file(AREA_PROG/'AOI_new.shp').to_crs(CRS_PROG)
AOI_GEO_PROG = [mapping(g) for g in AOI_PROG.geometry if g is not None]
TILES_PROG = [(r,c) for r in range(0,SHAPE_PROG[0],TILE_PROG)
                    for c in range(0,SHAPE_PROG[1],TILE_PROG)]
REFERENCIAS_PROG = defaultdict(list)
for camada,classe in [('Arvores','arvore'),('Grama','grama')]:
    refs = gpd.read_file(AREA_PROG/f'{camada}.gpkg').to_crs(CRS_PROG)
    linhas,colunas = rasterio.transform.rowcol(TRANSFORM_PROG,refs.geometry.x.to_numpy(),refs.geometry.y.to_numpy())
    for linha,coluna in zip(linhas,colunas):
        if 0<=linha<SHAPE_PROG[0] and 0<=coluna<SHAPE_PROG[1]:
            REFERENCIAS_PROG[(int(linha)//TILE_PROG*TILE_PROG,
                              int(coluna)//TILE_PROG*TILE_PROG)].append((int(linha),int(coluna),classe))
def caminho_tile_prog(idx): return DIR_TILES / f'tile_{idx:04d}.npz'
def caminho_skip_prog(idx): return DIR_TILES / f'tile_{idx:04d}.skip'
def caminho_spatial_prog(idx): return DIR_SPATIAL / f'spatial_{idx:04d}.npz'
def salvar_npz_atomico_prog(destino, **arrays):
    temporario = destino.with_name(destino.name+'.tmp')
    with open(temporario,'wb') as f:
        np.savez_compressed(f,**arrays)
    os.replace(temporario,destino)
def um_tile_prog(idx):
    r0,c0 = TILES_PROG[idx]
    h,w = min(TILE_PROG,SHAPE_PROG[0]-r0),min(TILE_PROG,SHAPE_PROG[1]-c0)
    janela = Window(c0,r0,w,h)
    trans = window_transform(janela,TRANSFORM_PROG)
    dentro = geometry_mask(AOI_GEO_PROG,out_shape=(h,w),transform=trans,invert=True)
    if not dentro.any():
        caminho_skip_prog(idx).write_text('fora da AOI',encoding='utf-8')
        return 0
    with rasterio.open(AREA_PROG/'AOI_RGB_NEW.tif') as rgb:
        im_tile = np.moveaxis(rgb.read(window=janela),0,-1)
    with rasterio.open(AREA_PROG/'AOI_NEW_MULTI.tif') as mult:
        bandas = mult.read(window=janela)
    seg = quickshift(im_tile,kernel_size=3,max_dist=5,ratio=0.04).astype(np.int32)
    nseg = int(seg.max())+1
    flat,valid = seg.ravel(),dentro.ravel()
    count = np.bincount(flat[valid],minlength=nseg)
    def media(v):
        soma = np.bincount(flat[valid],weights=v.ravel()[valid],minlength=nseg)
        return np.divide(soma,count,out=np.full(nseg,np.nan),where=count>0)
    nir,red = bandas[3].astype(np.float32),bandas[0].astype(np.float32)
    ndvi = (nir-red)/(nir+red+1e-8)
    valor = rgb2hsv(np.moveaxis(bandas[:3],0,-1))[...,2].astype(np.float32)
    bordas = canny(nir/10000,sigma=1).astype(np.float32)
    feat3 = np.stack([media(valor),media(bordas),media(ndvi)],axis=1).astype(np.float32)
    oito = []
    for band in bandas:
        arr = band.astype(np.float64).ravel()
        ok = valid & (arr != 0)
        n = np.bincount(flat[ok],minlength=nseg)
        sx = np.bincount(flat[ok],weights=arr[ok],minlength=nseg)
        sx2 = np.bincount(flat[ok],weights=arr[ok]**2,minlength=nseg)
        m = np.divide(sx,n,out=np.full(nseg,np.nan),where=n>0)
        sd = np.sqrt(np.maximum(0,np.divide(sx2,n,out=np.full(nseg,np.nan),where=n>0)-m*m))
        oito.append(m);oito.append(sd)
    feat8 = np.column_stack([oito[i] for i in [0,2,4,6,1,3,5,7]]).astype(np.float32)
    rr,cc = np.indices((h,w))
    mr = media(rr.astype(np.float32))+r0+0.5
    mc = media(cc.astype(np.float32))+c0+0.5
    xs = TRANSFORM_PROG.c+TRANSFORM_PROG.a*mc+TRANSFORM_PROG.b*mr
    ys = TRANSFORM_PROG.f+TRANSFORM_PROG.d*mc+TRANSFORM_PROG.e*mr
    labels = []
    for r,c,classe in REFERENCIAS_PROG[(r0,c0)]:
        if dentro[r-r0,c-c0]:
            labels.append((int(seg[r-r0,c-c0]),0 if classe=='arvore' else 1))
    lseg = np.array([q[0] for q in labels],dtype=np.int32)
    lcls = np.array([q[1] for q in labels],dtype=np.int8)
    if nseg>65535: raise RuntimeError('Número inesperado de segmentos (>65535): aumente a representação dos IDs.')
    salvar_npz_atomico_prog(caminho_tile_prog(idx),seg=seg.astype(np.uint16),
        dentro=dentro.astype(np.uint8),count=count.astype(np.int32),
        f3=feat3,f8=feat8,x=xs.astype(np.float64),y=ys.astype(np.float64),
        lseg=lseg,lcls=lcls)
    del im_tile,bandas,seg,ndvi,bordas,valor
    gc.collect()
    return int(np.count_nonzero(count))
def executar_lote_prog(max_tiles=TILES_POR_RODADA):
    feitos=0
    for idx in range(len(TILES_PROG)):
        if caminho_tile_prog(idx).exists() or caminho_skip_prog(idx).exists(): continue
        n = um_tile_prog(idx)
        feitos += 1
        print(f'Quadrante {idx+1}/{len(TILES_PROG)}: {n} segmentos',flush=True)
        if feitos>=max_tiles: break
    prontos=sum(caminho_tile_prog(i).exists() or caminho_skip_prog(i).exists()
                for i in range(len(TILES_PROG)))
    print(f'\nCHECKPOINT: {prontos}/{len(TILES_PROG)} quadrantes concluídos.')
    if prontos<len(TILES_PROG): print('Repita a ETAPA A; recomeçará do próximo quadrante.')
    else: print('ETAPA A CONCLUÍDA: execute a ETAPA B.')
    return prontos==len(TILES_PROG)
print(f'Preparado: {len(TILES_PROG)} quadrantes, {SHAPE_PROG[0]}x{SHAPE_PROG[1]} pixels, {len(REFERENCIAS_PROG)} quadrantes com referências.')
print('Rota progressiva independente. Não execute as células históricas que carregam o raster inteiro.')

In [ ]:
# ETAPA A — execute esta célula várias vezes, até concluir todos os quadrantes.
CONCLUIDO_A_PROG = executar_lote_prog(TILES_POR_RODADA)

In [ ]:
# ETAPA B — E1–E5 em dados REAIS dos quadrantes; execute SOMENTE após a ETAPA A.
import hashlib
if not all(caminho_tile_prog(i).exists() or caminho_skip_prog(i).exists()
           for i in range(len(TILES_PROG))):
    raise RuntimeError('A etapa A ainda não terminou. Repita sua célula até concluir.')
DSEG_PROG = SAIDA_PROG/'segmentos.csv.gz'
DREF_PROG = SAIDA_PROG/'referencias.csv'
if not DSEG_PROG.exists() or not DREF_PROG.exists():
    linhas,pares = [],[]
    for ti in range(len(TILES_PROG)):
        f=caminho_tile_prog(ti)
        if not f.exists(): continue
        with np.load(f,allow_pickle=False) as z:
            bons=np.where((z['count']>0)&np.isfinite(z['f3']).all(axis=1))[0]
            for si in bons:
                linhas.append((ti*65536+int(si),ti,int(si),int(z['count'][si]),
                   float(z['x'][si]),float(z['y'][si]),*map(float,z['f3'][si]),
                   *map(float,z['f8'][si])))
            for si,cl in zip(z['lseg'],z['lcls']):
                pares.append((ti*65536+int(si),'arvore' if cl==0 else 'grama'))
    nomes=['id','tile','sid','n_pixels','x','y','mean_intensity','mean_edges','mean_ndvi',
           'R_Mean','G_Mean','B_Mean','N_Mean','R_Std','G_Std','B_Std','N_Std']
    pd.DataFrame(linhas,columns=nomes).to_csv(DSEG_PROG,index=False,compression='gzip')
    pd.DataFrame(pares,columns=['id','classe']).to_csv(DREF_PROG,index=False)
    del linhas,pares
segmentos_prog=pd.read_csv(DSEG_PROG)
pares_prog=pd.read_csv(DREF_PROG)
features_prog=['mean_intensity','mean_edges','mean_ndvi']
unicos_prog=pares_prog.groupby('id')['classe'].nunique()
conflitos_prog=set(unicos_prog.index[unicos_prog>1])
rotulos_prog=(pares_prog.loc[~pares_prog.id.isin(conflitos_prog)]
              .drop_duplicates('id').merge(segmentos_prog,on='id',validate='one_to_one'))
rotulos_prog=rotulos_prog.replace([np.inf,-np.inf],np.nan).dropna(subset=features_prog)
rotulos_prog.to_csv(SAIDA_PROG/'rotulos_sem_duplicacao.csv',index=False)
print('E1 | rótulos brutos:',len(pares_prog),'únicos:',len(rotulos_prog),
      'conflitos:',len(conflitos_prog))
if len(rotulos_prog)<20 or rotulos_prog.classe.nunique()!=2:
    raise RuntimeError('Não há referências suficientes para as duas classes nos quadrantes.')
X_prog=rotulos_prog[features_prog].to_numpy(dtype=float)
y_prog=rotulos_prog.classe.to_numpy()
linhas_e1_prog=[]
for seed in [11,21,42,73,97]:
    tr,te=train_test_split(np.arange(len(y_prog)),test_size=.2,random_state=seed,stratify=y_prog)
    rf=RandomForestClassifier(n_estimators=500,random_state=42,n_jobs=2)
    rf.fit(X_prog[tr],y_prog[tr]); pred=rf.predict(X_prog[te])
    linhas_e1_prog.append({'semente':seed,'n_treino':len(tr),'n_teste':len(te),
       'f1_macro':f1_score(y_prog[te],pred,average='macro'),
       'acuracia_balanceada':balanced_accuracy_score(y_prog[te],pred)})
pd.DataFrame(linhas_e1_prog).to_csv(SAIDA_PROG/'E1_auditoria.csv',index=False)
auditoria_prog = {
  'referencias_brutas': int(len(pares_prog)),
  'segmentos_rotulados_distintos': int(pares_prog.id.nunique()),
  'registros_repetidos': int(len(pares_prog)-pares_prog.id.nunique()),
  'segmentos_conflitantes': int(len(conflitos_prog)),
  'segmentos_rotulados_validos': int(len(rotulos_prog)),
  'escopo': 'objetos de quadrantes independentes, nao segmentacao do raster integral'
}
(SAIDA_PROG/'E1_diagnostico.json').write_text(
    json.dumps(auditoria_prog,indent=2,ensure_ascii=False),encoding='utf-8')
# E2, E3, E4: blocos independentes, exclusão de centroides num raio de 25 m.
# Diferente do embargo geométrico por polígonos usado no código E2 original.
from scipy.spatial import cKDTree
resultados_prog=[];particoes_prog=[];descartadas_prog=[]
combos_prog=[c for k in [1,2,3] for c in itertools.combinations(features_prog,k)]
for tam in [250,500]:
    cx=rotulos_prog.x.to_numpy();cy=rotulos_prog.y.to_numpy()
    gx=np.floor((cx-cx.min())/tam).astype(int);gy=np.floor((cy-cy.min())/tam).astype(int)
    grupos=np.array([f'{a}_{b}' for a,b in zip(gx,gy)])
    if len(set(grupos))<5: continue
    for rodada,(tr,te) in enumerate(GroupShuffleSplit(n_splits=5,test_size=.2,random_state=42)
                                     .split(X_prog,y_prog,groups=grupos),1):
        distancia,_=cKDTree(np.column_stack([cx[te],cy[te]])).query(np.column_stack([cx[tr],cy[tr]]))
        tr=tr[distancia>25]
        if len(tr)<15 or len(te)<5 or len(set(y_prog[tr]))<2 or len(set(y_prog[te]))<2:
            descartadas_prog.append({'bloco_m':tam,'rodada':rodada,'motivo':'amostras insuficientes'})
            continue
        particoes_prog.append({'bloco_m':tam,'rodada':rodada,'n_treino':len(tr),'n_teste':len(te)})
        for combo in combos_prog:
            inds=[features_prog.index(s) for s in combo]; xx=X_prog[:,inds]
            modelos={
                'Random Forest':RandomForestClassifier(n_estimators=500,random_state=42,n_jobs=2),
                'SVM RBF':make_pipeline(StandardScaler(),SVC(kernel='rbf')),
                'Regressao Logistica':make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=42))
            }
            if len(combo)==3: modelos['Dummy']=DummyClassifier(strategy='most_frequent')
            for nome,mdl in modelos.items():
                mdl.fit(xx[tr],y_prog[tr]); pp=mdl.predict(xx[te])
                resultados_prog.append({'bloco_m':tam,'rodada':rodada,'modelo':nome,
                  'atributos':'+'.join(combo),'n_treino':len(tr),'n_teste':len(te),
                  'f1_macro':f1_score(y_prog[te],pp,average='macro',zero_division=0),
                  'acuracia_balanceada':balanced_accuracy_score(y_prog[te],pp)})
        print('E2–E4 | blocos:',tam,'rodada:',rodada,'concluída',flush=True)
pd.DataFrame(particoes_prog).to_csv(SAIDA_PROG/'E2_particoes.csv',index=False)
pd.DataFrame(descartadas_prog,columns=['bloco_m','rodada','motivo']).to_csv(SAIDA_PROG/'E2_descartadas.csv',index=False)
pd.DataFrame(resultados_prog).to_csv(SAIDA_PROG/'E3_E4_comparacoes.csv',index=False)
# RF completo para previsão exploratória por objeto; e clusters globais auxiliares.
rf_final_prog=RandomForestClassifier(n_estimators=500,random_state=42,n_jobs=2).fit(X_prog,y_prog)
joblib.dump(rf_final_prog,SAIDA_PROG/'rf_area1_lotes.joblib')
oito_prog=['R_Mean','G_Mean','B_Mean','N_Mean','R_Std','G_Std','B_Std','N_Std']
limpos_prog=segmentos_prog.dropna(subset=oito_prog).copy()
amostra8_prog=limpos_prog[oito_prog].to_numpy(float)
clusters_e5_prog={}
resumos_e5_prog=[]
for k in [4,6,8]:
    mdl=KMeans(n_clusters=k,random_state=42,n_init=10)
    clusters_e5_prog[k]=mdl.fit_predict(amostra8_prog)
    resumos_e5_prog.append({'K':k,'total_segmentos':len(amostra8_prog),
       'silhouette_amostra_500':float(silhouette_score(amostra8_prog,clusters_e5_prog[k],sample_size=min(500,len(amostra8_prog)),random_state=42))})
    if k==6: joblib.dump(mdl,SAIDA_PROG/'kmeans6_area1_lotes.joblib')
pd.DataFrame(resumos_e5_prog).to_csv(SAIDA_PROG/'E5_kmeans.csv',index=False)
# E5a: cinco configurações Quickshift nas mesmas até quatro janelas de 256 px.
# Não modifica qualquer segmento salvo em A.
linhas_e5_qprog=[]
janelas_referencia_prog=list(REFERENCIAS_PROG.keys())[:4]
with rasterio.open(AREA_PROG/'AOI_RGB_NEW.tif') as src_e5:
    for indice_e5,(r0_e5,c0_e5) in enumerate(janelas_referencia_prog,1):
        w_e5=Window(c0_e5,r0_e5,min(256,SHAPE_PROG[1]-c0_e5),
                    min(256,SHAPE_PROG[0]-r0_e5))
        img_e5=np.moveaxis(src_e5.read(window=w_e5),0,-1)
        for ks_e5,md_e5,ra_e5 in [(2,5,.04),(3,3,.04),(3,5,.04),(3,8,.04),(4,5,.04)]:
            inicio_e5=time.perf_counter()
            s_e5=quickshift(img_e5,kernel_size=ks_e5,max_dist=md_e5,ratio=ra_e5)
            linhas_e5_qprog.append({'janela':indice_e5,'r0':r0_e5,'c0':c0_e5,
                'kernel_size':ks_e5,'max_dist':md_e5,'ratio':ra_e5,
                'n_segmentos':int(np.unique(s_e5).size),
                'tempo_s':time.perf_counter()-inicio_e5})
            del s_e5
pd.DataFrame(linhas_e5_qprog).to_csv(SAIDA_PROG/'E5_quickshift_piloto.csv',index=False)
print('E5a | execuções Quickshift no piloto:',len(linhas_e5_qprog))
print('ETAPA B CONCLUÍDA. Resultados de E1–E5 em:',SAIDA_PROG)
print('Estes resultados são da segmentação por quadrantes independentes, não da V2 integral original.')

In [ ]:
# ETAPA C — proximidade por quadrante com checkpoints; repetir até CONCLUÍDO.
from shapely.strtree import STRtree
from shapely.geometry import Point
from rasterio.features import shapes
if not (SAIDA_PROG/'rf_area1_lotes.joblib').exists():
    raise RuntimeError('Execute a ETAPA B antes da proximidade espacial.')
RF_PROG = joblib.load(SAIDA_PROG/'rf_area1_lotes.joblib')
POSTES_PROG = (gpd.read_file(AREA_PROG/'postes.gpkg').to_crs(CRS_PROG)
               .clip(AOI_PROG).drop_duplicates(subset='geometry').reset_index(drop=True))
POSTES_SINDEX_PROG = POSTES_PROG.sindex
def um_spatial_prog(idx):
    origem=caminho_tile_prog(idx); destino=caminho_spatial_prog(idx)
    if not origem.exists():
        salvar_npz_atomico_prog(destino,seg_id=np.array([],dtype=np.int64),
           area=np.array([],dtype=np.float32),cls=np.array([],dtype=np.int8),
           par_seg=np.array([],dtype=np.int64),par_post=np.array([],dtype=np.int32),
           par_dist=np.array([],dtype=np.float32))
        return 0
    r0,c0=TILES_PROG[idx]; h=min(TILE_PROG,SHAPE_PROG[0]-r0);w=min(TILE_PROG,SHAPE_PROG[1]-c0)
    janela=Window(c0,r0,w,h)
    with np.load(origem,allow_pickle=False) as z:
        seg=z['seg'].astype(np.int32);dentro=z['dentro'].astype(bool)
        count=z['count'];f3=z['f3']
    ativos=np.flatnonzero((count>0)&np.isfinite(f3).all(axis=1))
    area=count[ativos]*RES_PROG
    clas=np.where(RF_PROG.predict(f3[ativos])=='arvore',0,1).astype(np.int8)
    ids=(idx*65536+ativos).astype(np.int64)
    # Todos os pares objeto–poste até 5 m, contando segmentos e postes distintos depois.
    geom_tile=box(*rasterio.windows.bounds(janela,TRANSFORM_PROG))
    prox_post=POSTES_SINDEX_PROG.query(geom_tile.buffer(5.5))
    pares_seg=[];pares_post=[];pares_dist=[]
    if len(prox_post):
        # Poligonizar apenas objetos que entram em ao menos um dos 12 cenários E7 > 5.
        params=[(.5,.7),(.6,.8),(2.5,.4),(4.,.5),(.4,.9),(1.35,.72)]
        possiveis=np.zeros(len(ativos),dtype=bool)
        for a,b in params:
            possiveis|=((a*np.power(area,b)>5)|(a+area*b>5))
        elegiveis=ativos[possiveis]
        if len(elegiveis):
            masc=dentro & np.isin(seg,elegiveis)
            for raw,id_segmento in shapes(seg,mask=masc,transform=window_transform(janela,TRANSFORM_PROG)):
                si=int(id_segmento)
                poly=shape(raw)
                candidatos=POSTES_SINDEX_PROG.query(poly.buffer(5.001))
                for p in candidatos:
                    d=poly.distance(POSTES_PROG.geometry.iloc[int(p)])
                    if d<=5.000001:
                        pares_seg.append(idx*65536+si)
                        pares_post.append(int(p))
                        pares_dist.append(float(d))
    salvar_npz_atomico_prog(destino,seg_id=ids,area=area.astype(np.float32),cls=clas,
       par_seg=np.asarray(pares_seg,dtype=np.int64),
       par_post=np.asarray(pares_post,dtype=np.int32),
       par_dist=np.asarray(pares_dist,dtype=np.float32))
    del seg,dentro,f3;gc.collect()
    return len(pares_seg)
novos=0
for ti in range(len(TILES_PROG)):
    if caminho_spatial_prog(ti).exists(): continue
    n=um_spatial_prog(ti);novos+=1
    print(f'Espacial {ti+1}/{len(TILES_PROG)}: {n} pares objeto–poste',flush=True)
    if novos>=TILES_POR_RODADA: break
feitos=sum(caminho_spatial_prog(i).exists() for i in range(len(TILES_PROG)))
print(f'CHECKPOINT ESPACIAL: {feitos}/{len(TILES_PROG)} quadrantes.')
print('ETAPA C CONCLUÍDA: execute D.' if feitos==len(TILES_PROG) else 'Repita C para continuar do último quadrante.')

In [ ]:
# ETAPA D — agregação de E6/E7 e relatório E8; execute após completar a ETAPA C.
if not all(caminho_spatial_prog(i).exists() for i in range(len(TILES_PROG))):
    raise RuntimeError('Etapa C incompleta: repita a célula C até todos os quadrantes serem processados.')
seg_linhas=[];par_linhas=[]
for ti in range(len(TILES_PROG)):
    with np.load(caminho_spatial_prog(ti),allow_pickle=False) as z:
        seg_linhas.extend(zip(z['seg_id'].tolist(),z['area'].tolist(),z['cls'].tolist()))
        par_linhas.extend(zip(z['par_seg'].tolist(),z['par_post'].tolist(),z['par_dist'].tolist()))
seg_df_prog=pd.DataFrame(seg_linhas,columns=['id','area','classe_cod']).drop_duplicates('id')
pares_df_prog=pd.DataFrame(par_linhas,columns=['id','poste_id','distancia_m'])
if not pares_df_prog.empty:
    pares_df_prog=pares_df_prog.groupby(['id','poste_id'],as_index=False)['distancia_m'].min()
seg_df_prog.to_csv(SAIDA_PROG/'segmentos_preditos.csv.gz',index=False,compression='gzip')
pares_df_prog.to_csv(SAIDA_PROG/'pares_segmento_poste_5m.csv.gz',index=False,compression='gzip')
# E6: seleção C3 > 7 sem e com filtro previsto como árvore, e raios de 1,3,5 metros.
seg_df_prog['C3'] = 2.5*np.power(seg_df_prog['area'],.4)
linhas_e6=[]
for filtro in ['todos','arvore']:
    escolha=seg_df_prog.loc[(seg_df_prog.C3>7)&
             ((seg_df_prog.classe_cod==0) if filtro=='arvore' else True),'id']
    for raio in [1,3,5]:
        proximos=pares_df_prog.loc[(pares_df_prog.distancia_m<=raio)&
                                    (pares_df_prog.id.isin(escolha))]
        linhas_e6.append({'filtro':filtro,'distancia_m':raio,
            'segmentos_elegiveis':len(escolha),'segmentos_proximos':proximos.id.nunique(),
            'postes_proximos':proximos.poste_id.nunique()})
pd.DataFrame(linhas_e6).to_csv(SAIDA_PROG/'E6_postes.csv',index=False)
# E7: 12 expressões × 3 limiares × 2 filtros × 3 distâncias = 216 cenários.
params_prog=[(.5,.7),(.6,.8),(2.5,.4),(4.,.5),(.4,.9),(1.35,.72)]
base=set(seg_df_prog.loc[seg_df_prog.C3>7,'id'])
linhas_e7=[]
for j,(a,b) in enumerate(params_prog,1):
    for forma in ['potencial','linear']:
        score=a*np.power(seg_df_prog.area,b) if forma=='potencial' else a+seg_df_prog.area*b
        for limiar in [5,7,9]:
            for filtro in ['todos','arvore']:
                selecionados=set(seg_df_prog.loc[(score>limiar)&
                   ((seg_df_prog.classe_cod==0) if filtro=='arvore' else True),'id'])
                inter=len(selecionados&base); uniao=len(selecionados|base)
                jaccard=inter/uniao if uniao else float('nan')
                for raio in [1,3,5]:
                    prox=pares_df_prog.loc[(pares_df_prog.distancia_m<=raio)&
                                           (pares_df_prog.id.isin(selecionados))]
                    linhas_e7.append({'configuracao':f'C{j}','forma':forma,'limiar':limiar,
                       'filtro':filtro,'distancia_m':raio,'segmentos_elegiveis':len(selecionados),
                       'segmentos_proximos':prox.id.nunique(),'postes_proximos':prox.poste_id.nunique(),
                       'jaccard_vs_C3_potencial_7':jaccard})
e7df=pd.DataFrame(linhas_e7)
assert len(e7df)==216
e7df.to_csv(SAIDA_PROG/'E7_alometria_216_cenarios.csv',index=False)
# Auditoria de qualidade de extrapolação: a base só tem rótulos árvore/grama.
# Uma predição 'arvore' em toda a AOI NÃO prova que o objeto é vegetação.
fracao_arvore_prog = float((seg_df_prog['classe_cod']==0).mean()) if len(seg_df_prog) else float('nan')
qtd_postes_prog = len(POSTES_PROG) if 'POSTES_PROG' in globals() else None
e6_arv1_prog = next((r for r in linhas_e6 if r['filtro']=='arvore' and r['distancia_m']==1),None)
fracao_postes_arv1_prog = (
    float(e6_arv1_prog['postes_proximos']/qtd_postes_prog)
    if e6_arv1_prog and qtd_postes_prog else None
)
avisos_qc_prog = []
if fracao_arvore_prog > .70:
    avisos_qc_prog.append('ALERTA: mais de 70% dos segmentos elegiveis foram previstos como arvore; verificar extrapolacao a fundo sem rotulo de classe nao vegetacao.')
if fracao_postes_arv1_prog is not None and fracao_postes_arv1_prog > .70:
    avisos_qc_prog.append('ALERTA: mais de 70% dos postes foram marcados com segmento previsto como arvore a 1 m; isso NAO constitui indicador validado de risco.')
qcdados_prog = {'fracao_segmentos_previstos_arvore':fracao_arvore_prog,
    'postes_na_aoi':qtd_postes_prog,
    'fracao_postes_com_segmento_arvore_ate_1m':fracao_postes_arv1_prog,
    'avaliacao_de_risco_eletrico_validada':False,
    'classe_nao_vegetacao_rotulada':False,
    'avisos':avisos_qc_prog}
(SAIDA_PROG/'E8_auditoria_qualidade.json').write_text(
    json.dumps(qcdados_prog,indent=2,ensure_ascii=False),encoding='utf-8')
if avisos_qc_prog:
    print('\n*** E8: ALERTA DE QUALIDADE ***')
    for aviso in avisos_qc_prog: print('-',aviso)
    print('Estes resultados sao exploratorios; nao publicar contagens como arvores reais ou risco eletrico.')
manifesto={
  'metodo':'segmentacao por quadrantes Quickshift 256x256, checkpoint/resume; não equivalente ao raster integral',
  'raster_pixels':int(SHAPE_PROG[0]*SHAPE_PROG[1]),
  'total_quadrantes':len(TILES_PROG),
  'quadrantes_validos':sum(caminho_tile_prog(i).exists() for i in range(len(TILES_PROG))),
  'segmentos_validos':len(seg_df_prog),
  'rotulos_sem_conflitos':len(pd.read_csv(SAIDA_PROG/'rotulos_sem_duplicacao.csv')),
  'pares_objeto_poste_5m':len(pares_df_prog),
  'cenarios_e7':len(e7df),
  'controle_qualidade': qcdados_prog,
  'limitacoes':['Limites dos quadrantes podem dividir copas e alterar objetos.',
      'A extração de atributos é por pixel dentro da AOI e em janelas alinhadas aos rasters.',
      'A etapa E2/E3/E4 usa embargo de 25 m entre centroides, não entre polígonos.',
      'Não se usa o cluster 3 histórico para selecionar a base de treinamento.',
      'C1–C6 são índices empíricos sem validação por alturas observadas.',
      'Proximidade horizontal a postes não comprova contato com cabos ou risco elétrico.',
      'Desempenho não pode ser comparado diretamente à acurácia histórica 0,87.'],
  'arquivos_resultados':['E8_auditoria_qualidade.json','E1_auditoria.csv','E2_particoes.csv','E3_E4_comparacoes.csv',
    'E5_kmeans.csv','E5_quickshift_piloto.csv','E6_postes.csv','E7_alometria_216_cenarios.csv']}
(SAIDA_PROG/'E8_manifesto.json').write_text(json.dumps(manifesto,indent=2,ensure_ascii=False),encoding='utf-8')
relatorio = [
'# Execução progressiva — resultados reais da Area1 por quadrantes',
'','**Não é a execução integral do algoritmo original aplicado ao raster inteiro.**','',
f"Quadrantes: {manifesto['quadrantes_validos']} / {manifesto['total_quadrantes']}",
f"Segmentos válidos: {manifesto['segmentos_validos']}",
f"Referências distintas: {manifesto['rotulos_sem_conflitos']}",
f"Cenários E7: {manifesto['cenarios_e7']}",'',
'## E6 — proximidade a postes','',
pd.DataFrame(linhas_e6).to_markdown(index=False),'',
'## Controle de qualidade e limitações','',*['- '+a for a in avisos_qc_prog],*['- '+v for v in manifesto['limitacoes']]]
(SAIDA_PROG/'E8_relatorio.md').write_text('\n'.join(relatorio),encoding='utf-8')
print('ETAPA D CONCLUÍDA — confira os CSV, E8_manifesto.json e E8_relatorio.md em:')
print(SAIDA_PROG)
print('Resultados experimentais por QUADRANTES: não declarar equivalentes à V2 integral histórica.')

# Segmentação de arvores e análise

## Preparação dos dados para segmentação:

Instalação das bibliotecas neessárias:

In [ ]:
!pip install rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Importação das funções e pacotes

In [ ]:
from skimage.segmentation import felzenszwalb, slic, quickshift, watershed
from skimage.segmentation import mark_boundaries
from skimage.util import img_as_float
from skimage.filters import sobel
from skimage.color import rgb2gray
from skimage import color
import rasterio
from matplotlib import pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
from rasterio.plot import show

Abrir a imagem com o rasterio, convertendo num numpy array:

In [ ]:
path_img = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_RGB_NEW.tif'
src_rgb = rasterio.open(path_img)
im = src_rgb.read()
im  = im.transpose([1,2,0])
'''R = src_rgb.read(3)
G = src_rgb.read(2)
B = src_rgb.read(1)


R = R / 10000
G = G / 10000
B = B / 10000

R = R * 255
G = G * 255
B = B * 255

R = R.astype('uint8')
G = G.astype('uint8')
B = B.astype('uint8')

im = np.dstack((R,G,B))'''

Plotando a imagem com o matplotlib:

In [ ]:
plt.figure(figsize=[12,12])
plt.imshow(im)
plt.axis('off')

Abrir o poligono da área de interesse:

In [ ]:
AOI = gpd.read_file('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_new.shp')

## Segmentação:

Aplicação do quickshift (Ver artigos relacionados)

In [ ]:
segments_quick2 = quickshift(im, kernel_size=3, max_dist=5, ratio=0.04)

In [ ]:
plt.figure(figsize=[16,16])
plt.imshow(mark_boundaries(im, segments_quick2))
plt.axis('off')

In [ ]:
segments_quick2 = segments_quick2.astype('uint16')

Conversão para poligono e extração das caracteristicas espectrais:

In [ ]:
from rasterio import features
from shapely.geometry import shape
import geopandas as gpd
import shapely
import json
from rasterio.mask import mask
import pandas as pd

Primeiro, converter os segmentos em poligonos usando geopandas e rasterio:

In [ ]:
find_shapes = features.shapes(segments_quick2, transform=src_rgb.transform)

In [ ]:
polygons = [shapely.geometry.Polygon(shape[0]["coordinates"][0]) for shape in find_shapes]

In [ ]:
crs = src_rgb.crs
Poly_gdf = gpd.GeoDataFrame(crs=crs, geometry=polygons)

Salvar os poligonos em um JSON:

In [ ]:
Poly_gdf.to_file("polygons.json")

In [ ]:
Poly_gdf.boundary.plot()

Obter somente os poligonos da área de interesse:

In [ ]:
clip_poly = Poly_gdf.clip(AOI)

In [ ]:
clip_poly.boundary.plot()

Extrair a média espectral de cada segmento:

In [ ]:
clip_poly = clip_poly.reset_index(drop=True)

In [ ]:
json_dataset = json.loads(clip_poly.to_json())['features']

In [ ]:
path_img_mult = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_NEW_MULTI.tif'
src = rasterio.open(path_img_mult)

X = []
Y = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=0)

  out_image = out_image.transpose([1,2,0])

  R_mean = out_image[:,:,0][np.nonzero(out_image[:,:,0])].mean()
  G_mean = out_image[:,:,1][np.nonzero(out_image[:,:,1])].mean()
  B_mean = out_image[:,:,2][np.nonzero(out_image[:,:,2])].mean()
  N_mean = out_image[:,:,3][np.nonzero(out_image[:,:,3])].mean()
  R_std = out_image[:,:,0][np.nonzero(out_image[:,:,0])].std()
  G_std = out_image[:,:,1][np.nonzero(out_image[:,:,1])].std()
  B_std = out_image[:,:,2][np.nonzero(out_image[:,:,2])].std()
  N_std = out_image[:,:,3][np.nonzero(out_image[:,:,3])].std()




  X.append([R_mean,G_mean,B_mean,N_mean,R_std,G_std,B_std,N_std])

In [ ]:
X = np.array(X)

## Aplicação do K-Means

Geramos o dataframe com a informação das médias espectrais por segmentos:

In [ ]:
df = pd.DataFrame(X, columns=['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std'])

In [ ]:
df

In [ ]:
df.dropna(inplace=True)

Aplicamos o Kmeans:

In [ ]:
from sklearn.cluster import KMeans

# Apply K-Means clustering
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10) # You can change the number of clusters
kmeans_clusters = kmeans.fit_predict(df[['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std']])

# Add K-Means cluster labels to the DataFrame
df['kmeans_cluster'] = kmeans_clusters

display(df.head())

Aplicamos o resultado do Kmeans para os segmentos e visualizamos:

In [ ]:
clip_poly['kmeans_cluster'] = df['kmeans_cluster']

In [ ]:
import matplotlib.pyplot as plt

# Plot the GeoDataFrame, colored by cluster
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
clip_poly.plot(column='kmeans_cluster', ax=ax, legend=True, cmap='magma') # Changed colormap to 'viridis'
plt.title('Polygons Colored by kmeans_cluster')
plt.axis('off')
plt.show()

Salvamos o resultado em um JSON:

In [ ]:
Poly_gdf.to_file('polygons.json')

In [ ]:
clip_poly[clip_poly['kmeans_cluster'] == 3].to_file('polygons_arvores.json')

Selecionamos somente o cluster de arvores:

In [ ]:
import rasterio.plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
clip_poly[clip_poly['kmeans_cluster'] == 3].boundary.plot(ax=ax, color='red')
plt.title('Original Image')
plt.axis('off')
plt.show()

In [ ]:
path_grama = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Grama.gpkg'
path_arvore = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Arvores.gpkg'

In [ ]:
gdf_grama = gpd.read_file(path_grama)
gdf_arvore = gpd.read_file(path_arvore)

In [ ]:
gdf_grama['Classe'] = 'grama'
gdf_arvore['Classe'] = 'arvore'

In [ ]:
gdf_full = pd.concat([gdf_grama, gdf_arvore])
gdf_full.to_crs(clip_poly.crs, inplace=True)

In [ ]:
gdf_arvore_seg = clip_poly[clip_poly['kmeans_cluster'] == 3]

In [ ]:
gdf_arvore_seg_filter = gdf_arvore_seg.sjoin(gdf_full, how='inner', predicate='intersects')

In [ ]:
gdf_arvore_seg_filter.plot()

In [ ]:
gdf_arvore_seg_filter

In [ ]:
src = rasterio.open(path_img_mult)

In [ ]:
img = src.read()

In [ ]:
img.shape

In [ ]:
nir = img[3]
red = img[0]

In [ ]:
nir.shape

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(nir, cmap='gray')
plt.title('NIR Band')
plt.axis('off')
plt.show()

In [ ]:
import numpy as np

# Calculate NDVI
# Avoid division by zero by adding a small epsilon to the denominator
ndvi = (nir.astype(float) - red.astype(float)) / (nir.astype(float) + red.astype(float) + 1e-8)

# Plot NDVI
plt.figure(figsize=(10, 10))
plt.imshow(ndvi, cmap='RdYlGn', vmin=0, vmax=1) # RdYlGn is a common colormap for NDVI
plt.colorbar(label='NDVI Value')
plt.title('Normalized Difference Vegetation Index (NDVI)')
plt.axis('off')
plt.show()

In [ ]:
nir.max()

In [ ]:
from skimage.feature import canny
from skimage.util import img_as_ubyte

# Ensure nir is in the correct format (uint8) for Canny
nir_uint = nir/10000

# Apply Canny edge detection
edges = canny(nir_uint, sigma=1)

# Plot the edges
plt.figure(figsize=(10, 10))
plt.imshow(edges, cmap='gray')
plt.title('Canny Edges of NIR Band')
plt.axis('off')
plt.show()

In [ ]:
img_rgb = img[0:3]

In [ ]:
img_rgb = img_rgb.transpose([1,2,0])

In [ ]:
from skimage.color import rgb2hsv

# Convert RGB to HSV
img_hsv = rgb2hsv(img_rgb)

# Extract the intensity (Value) channel, which is the third channel (index 2)
intensity = img_hsv[:, :, 2]

# Display the intensity image
plt.figure(figsize=(10, 10))
plt.imshow(intensity, cmap='gray')
plt.title('Intensity Channel (HSV)')
plt.axis('off')
plt.show()

In [ ]:
# Convert edges (boolean) to float for stacking
edges_float = edges.astype(np.float32)

# Stack intensity and edges as two separate bands
# The order here is intensity first, then edges
stacked_bands = np.stack([intensity, edges_float, ndvi], axis=0)

# Get metadata from the original source (src) for writing the new TIFF
profile = src.profile

# Update profile for the new stacked image
# Change the number of bands to 2 and data type to float32
profile.update(
    dtype=rasterio.float32,
    count=3,
    nodata=None
)

# Define the output path for the new TIFF file
output_filepath = 'features_stack.tif'

# Write the stacked image to a new TIFF file
with rasterio.open(output_filepath, 'w', **profile) as dst:
    dst.write(stacked_bands)


In [ ]:
src = rasterio.open(output_filepath)
json_dataset = json.loads(gdf_arvore_seg_filter.to_json())['features']

X = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=-999)
  classe_values = json_dataset[i]['properties']['Classe']

  mean_intensity = out_image[0][out_image[0] != -999].mean()
  mean_edges = out_image[1][out_image[1] != -999].mean()
  mean_ndvi = out_image[2][out_image[2] != -999].mean()
  X.append([mean_intensity, mean_edges, mean_ndvi,classe_values])


In [ ]:
df = pd.DataFrame(X, columns=['mean_intensity','mean_edges','mean_ndvi','classe'])

In [ ]:
import seaborn as sns

# Create a pairplot with 'classe' as hue
sns.pairplot(df, hue='classe')
plt.show()

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Separate features (X) and target (y)
X = df.drop('classe', axis=1) # Features are all columns except 'classe'
y = df['classe'] # Target is the 'classe' column

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Random Forest Classifier
# You can adjust parameters like n_estimators (number of trees) as needed
rf_classifier = RandomForestClassifier(n_estimators=500, random_state=42)

# Train the model
rf_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test)

# Evaluate the model
print("Random Forest Classifier Report:")
print(classification_report(y_test, y_pred))
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}")

In [ ]:
# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=rf_classifier.classes_)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_classifier.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

In [ ]:
src = rasterio.open(output_filepath)
json_dataset = json.loads(gdf_arvore_seg.to_json())['features']

X_pred = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=-999)

  mean_intensity = out_image[0][out_image[0] != -999].mean()
  mean_edges = out_image[1][out_image[1] != -999].mean()
  mean_ndvi = out_image[2][out_image[2] != -999].mean()
  X_pred.append([mean_intensity, mean_edges, mean_ndvi])

In [ ]:
df_pred = pd.DataFrame(X_pred, columns=['mean_intensity','mean_edges','mean_ndvi'])

In [ ]:
gdf_arvore_seg['predicted_classe'] = rf_classifier.predict(df_pred)

In [ ]:
gdf_arvore_seg

In [ ]:
gdf_arvore_seg['area'] = gdf_arvore_seg.area

In [ ]:
tree_polys = gdf_arvore_seg.copy()

## Seleção das arvores e aplicação da Alometria:

Selecionamos os segmentos que pertence ao cluster das arvores:

In [ ]:
tree_polys

Aplicamos a formula da alometria:

In [ ]:
import numpy as np

#C1
a1 = 0.5
b1 = 0.7
tree_polys['Power_tree_size_C1'] = a1 * (tree_polys['area'] ** b1)
tree_polys['Linear_tree_size_C1'] = a1 + (tree_polys['area'] * b1)


#C2
a2 = 0.6
b2 = 0.8
tree_polys['Power_tree_size_C2'] = a2 * (tree_polys['area'] ** b2)
tree_polys['Linear_tree_size_C2'] = a2 + (tree_polys['area'] * b2)


#C3
a3 = 2.5
b3 = 0.4
tree_polys['Power_tree_size_C3'] = a3 * (tree_polys['area'] ** b3)
tree_polys['Linear_tree_size_C3'] = a3 + (tree_polys['area'] * b3)

#C4
a4 = 4
b4 = 0.5
tree_polys['Power_tree_size_C4'] = a4 * (tree_polys['area'] ** b4)
tree_polys['Linear_tree_size_C4'] = a4 + (tree_polys['area'] * b4)

#C5

a5 = 0.4
b5 = 0.9
tree_polys['Power_tree_size_C5'] = a5 * (tree_polys['area'] ** b5)
tree_polys['Linear_tree_size_C5'] = a5 + (tree_polys['area'] * b5)



a6 = 1.35
b6 = 0.72
tree_polys['Power_tree_size_C6'] = a6 * (tree_polys['area'] ** b6)
tree_polys['Linear_tree_size_C6'] = a6 + (tree_polys['area'] * b6)

In [ ]:
columns = ['Power_tree_size_C1', 'Linear_tree_size_C1', 'Power_tree_size_C2', 'Linear_tree_size_C2',
           'Power_tree_size_C3', 'Linear_tree_size_C3', 'Power_tree_size_C4', 'Linear_tree_size_C4',
           'Power_tree_size_C5', 'Linear_tree_size_C5', 'Power_tree_size_C6', 'Linear_tree_size_C6']

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(data=tree_polys[columns])
plt.title('Box Plot of Tree Size Metrics for Different Constants')
plt.ylabel('Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(data=tree_polys['Power_tree_size_C3'])
plt.title('Box Plot of Tree Size Metrics for C3')
plt.ylabel('Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(tree_polys['Power_tree_size_C3'], bins=30, edgecolor='black')
plt.title('Histogram of Power_tree_size_C3')
plt.xlabel('Power_tree_size_C3')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

Obtemos somente arvores maiores que 7 metros:

In [ ]:
filter_tree = tree_polys[tree_polys['Power_tree_size_C3'] > 7]

Salvamos em um arquivo JSON:

In [ ]:
filter_tree.to_file('arvores_segmentadas.json')

## Análise das arvores proximas a linhas e postes:

Importação e visualização das linhas e postes:

In [ ]:
Postes = gpd.read_file('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/postes.gpkg')

Arvores maiores que 7m a 1m de distancia ou menos de postes:

In [ ]:
Postes = Postes.to_crs(filter_tree.crs)

In [ ]:
Postes = Postes.clip(AOI)

### Arvores e postes a 1m

In [ ]:
postes_prox_tree_1m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=1)

In [ ]:
arvores_prox_postes_1m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_1m.boundary.plot(ax=ax, color='green')
postes_prox_tree_1m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

### Arvores e postes a 3m

In [ ]:
postes_prox_tree_3m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=3)

In [ ]:
arvores_prox_postes_3m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=3)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_3m.boundary.plot(ax=ax, color='green')
postes_prox_tree_3m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

### Arvores e postes a 5m

In [ ]:
postes_prox_tree_5m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=5)

In [ ]:
arvores_prox_postes_5m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=5)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_5m.boundary.plot(ax=ax, color='green')
postes_prox_tree_5m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

## V2 — Auditoria de amostras e avaliação sem duplicação de segmentos

**Etapa 2.** As células anteriores preservam integralmente o código original; apenas as saídas antigas foram limpas nesta cópia experimental (continuam no notebook original). Execute as células anteriores para reconstruir `df` e `gdf_arvore_seg_filter`, depois execute as células a seguir.

A auditoria procura (1) segmentos repetidos pelo `sjoin(intersects)`, (2) rótulos conflitantes e (3) sobreposição de identificadores de segmentos entre treino e teste original. O novo teste é exploratório, usa somente `Area1`, exclui explicitamente os segmentos com rótulos conflitantes, mantém uma linha por segmento e executa cinco separações por grupo. Não permite inferir risco elétrico, nem resolve autocorrelação espacial entre segmentos vizinhos. Nenhum resultado novo foi executado ou validado ao publicar este notebook.


In [ ]:
# V2 — auditoria da independência das amostras (não modifica o RF original)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

atributos_v2 = ["mean_intensity", "mean_edges", "mean_ndvi"]
assert all(c in df.columns for c in atributos_v2 + ["classe"])
assert len(df) == len(gdf_arvore_seg_filter), "Linhas desalinhadas entre df e GeoDataFrame."
assert np.array_equal(df["classe"].astype(str).to_numpy(),
                      gdf_arvore_seg_filter["Classe"].astype(str).to_numpy()), (
    "Classes ou ordem das linhas não coincidem; interrompa a auditoria."
)

# O spatial join preserva o índice do segmento da tabela à esquerda.
segmentos_v2 = pd.Series(gdf_arvore_seg_filter.index.to_numpy(), name="segmento")
pares_v2 = pd.DataFrame({
    "segmento": segmentos_v2,
    "classe": df["classe"].astype(str).reset_index(drop=True)
})
conflitos_v2 = pares_v2.groupby("segmento")["classe"].nunique()
ids_conflitantes_v2 = conflitos_v2[conflitos_v2 > 1].index

indices_tr_v2, indices_te_v2 = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42
)
ids_em_ambos_v2 = set(segmentos_v2.iloc[indices_tr_v2]) & set(segmentos_v2.iloc[indices_te_v2])
print("=== Diagnóstico da divisão original ===")
print("Pares objeto-rótulo:", len(df))
print("Segmentos únicos:", pares_v2["segmento"].nunique())
print("Registros repetidos por segmento:", len(df) - pares_v2["segmento"].nunique())
print("Segmentos com classes conflitantes:", len(ids_conflitantes_v2))
print("Segmentos compartilhados no treino e teste originais:", len(ids_em_ambos_v2))
print("Registros de teste desses segmentos:", int(
    segmentos_v2.iloc[indices_te_v2].isin(ids_em_ambos_v2).sum()))
print("Nota: segmentos vizinhos podem continuar espacialmente correlacionados.")


In [ ]:
# V2 — teste exploratório: 1 linha por segmento, cinco divisões agrupadas
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

pares_unicos_v2 = pares_v2.loc[
    ~pares_v2["segmento"].isin(ids_conflitantes_v2)
].drop_duplicates("segmento")
print("Registros excluídos (repetições/conflitos):", len(df) - len(pares_unicos_v2))
X_agr_v2 = df.loc[pares_unicos_v2.index, atributos_v2].to_numpy(dtype=float)
y_agr_v2 = pares_unicos_v2["classe"].to_numpy()
g_agr_v2 = pares_unicos_v2["segmento"].to_numpy()
if len(pares_unicos_v2) < 10 or len(np.unique(y_agr_v2)) < 2:
    raise ValueError("Não há segmentos distintos suficientes com ambas as classes.")
if not np.isfinite(X_agr_v2).all():
    raise ValueError("Atributos não finitos; reveja as extrações antes da avaliação.")

linhas_v2 = []
cv_v2 = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
for rodada, (tr, te) in enumerate(cv_v2.split(X_agr_v2, y_agr_v2, groups=g_agr_v2), start=1):
    if len(np.unique(y_agr_v2[tr])) < 2 or len(np.unique(y_agr_v2[te])) < 2:
        print(f"Rodada {rodada} ignorada: falta classe no treino ou no teste.")
        continue
    modelo_auditoria_v2 = RandomForestClassifier(
        n_estimators=500, random_state=42, n_jobs=-1)
    modelo_auditoria_v2.fit(X_agr_v2[tr], y_agr_v2[tr])
    pred_v2 = modelo_auditoria_v2.predict(X_agr_v2[te])
    linhas_v2.append({
        "rodada": rodada,
        "n_treino": len(tr),
        "n_teste": len(te),
        "acuracia": accuracy_score(y_agr_v2[te], pred_v2),
        "acuracia_balanceada": balanced_accuracy_score(y_agr_v2[te], pred_v2),
        "f1_macro": f1_score(y_agr_v2[te], pred_v2, average="macro", zero_division=0)
    })
resultado_auditoria_v2 = pd.DataFrame(linhas_v2)
if resultado_auditoria_v2.empty:
    print("Não houve nenhuma rodada válida; métricas não disponíveis.")
else:
    print(resultado_auditoria_v2.to_string(index=False))
    print("\nMédia e desvio-padrão (rodadas válidas):")
    print(resultado_auditoria_v2[["acuracia", "acuracia_balanceada", "f1_macro"]]
          .agg(["mean", "std"]).to_string())
    print("\nATENÇÃO: não comparar diretamente com 0,87, pois os testes diferem.")


## E2 — validação espacial por blocos com a mesma Area1

**Etapa seguinte à auditoria E1.** Treino e teste agora são separados em *blocos geográficos*, não apenas por registros. Uma faixa de exclusão de **25 metros** ao redor dos polígonos de teste evita que amostras de treinamento fiquem imediatamente encostadas nos objetos avaliados.

Usa somente `df` e `gdf_arvore_seg_filter` produzidos acima (imagens/rótulos existentes na `Area1`). Remove segmentos repetidos e aqueles com rótulos conflitantes, como no E1, preservando a classificação original e seus resultados históricos. O RF mantém **500 árvores**, `random_state=42` e os três atributos originais.

Testa blocos de **250 m** e **500 m**, com até 10 partições por escala. Cada partição contém blocos inteiros no teste; se não houver duas classes ou dados suficientes, a partição é descartada e isso é relatado. A proporção final de objetos em teste pode diferir de 20%, pois o sorteio é por blocos.

**Atenção:** separações dentro da mesma área de interesse são validação espacial interna, não teste de generalização para outra cidade. As métricas só passam a existir após executar as células com os dados reais; não são substitutas das métricas históricas de 0,87.


In [ ]:
# E2a — preparar dados com objetos únicos e geometria espacial real (Area1)
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score
)

campos_e2 = ["mean_intensity", "mean_edges", "mean_ndvi"]
assert len(df) == len(gdf_arvore_seg_filter), "As linhas de df e do sjoin não coincidem."
assert all(x in df.columns for x in campos_e2 + ["classe"])
assert "Classe" in gdf_arvore_seg_filter.columns
assert gdf_arvore_seg_filter.crs is not None, "É necessário um CRS conhecido."
rotulos_df_e2 = df["classe"].astype(str).str.strip().str.lower().to_numpy()
rotulos_geo_e2 = gdf_arvore_seg_filter["Classe"].astype(str).str.strip().str.lower().to_numpy()
assert np.array_equal(rotulos_df_e2, rotulos_geo_e2), (
    "Ordem ou classes diferem entre df e gdf; auditoria interrompida."
)

dados_e2 = df[campos_e2].reset_index(drop=True).copy()
dados_e2["classe"] = rotulos_df_e2
# O índice à esquerda do sjoin é o identificador do segmento; pode se repetir.
dados_e2["segmento_id"] = gdf_arvore_seg_filter.index.astype(str).to_numpy()
dados_e2["geometry"] = list(gdf_arvore_seg_filter.geometry.to_numpy())
base_e2 = gpd.GeoDataFrame(
    dados_e2, geometry="geometry", crs=gdf_arvore_seg_filter.crs
)
n_classes_e2 = base_e2.groupby("segmento_id")["classe"].nunique()
ids_conflitantes_e2 = set(n_classes_e2[n_classes_e2 > 1].index)
qtd_pares_e2 = len(base_e2)
base_e2 = (base_e2.loc[~base_e2["segmento_id"].isin(ids_conflitantes_e2)]
           .drop_duplicates(subset=["segmento_id"]).copy())
for campo_e2 in campos_e2:
    base_e2[campo_e2] = pd.to_numeric(base_e2[campo_e2], errors="coerce")
base_e2[campos_e2] = base_e2[campos_e2].replace([np.inf, -np.inf], np.nan)
base_e2 = base_e2.dropna(subset=campos_e2 + ["geometry"]).reset_index(drop=True)
if base_e2.geometry.is_empty.any() or (~base_e2.geometry.is_valid).any():
    raise ValueError("Polígonos vazios ou inválidos; revisar geometrias antes da avaliação.")
# EPSG:31982 é o CRS métrico dos rasters originais da pesquisa.
base_e2 = base_e2.to_crs(epsg=31982)
if len(base_e2) < 20 or base_e2["classe"].nunique() != 2:
    raise ValueError("Dados insuficientes para avaliar as duas classes por blocos.")
print("=== E2 | Preparação para validação espacial ===")
print("Pares objeto-rótulo anteriores:", qtd_pares_e2)
print("Segmentos conflitantes excluídos:", len(ids_conflitantes_e2))
print("Objetos distintos e válidos:", len(base_e2))
print("Classes:", base_e2["classe"].value_counts().to_dict())
print("CRS métrico:", base_e2.crs)


In [ ]:
# E2b — separação espacial por blocos, embargo geométrico de 25 m e RF original
# Não baixa novos dados nem altera o modelo ou as variáveis do experimento histórico.
resultados_e2 = []
ignorados_e2 = []
tam_blocos_e2 = [250, 500]  # metros
faixa_exclusao_e2 = 25  # metros
centros_e2 = base_e2.geometry.centroid
xmin_e2, ymin_e2, _, _ = base_e2.total_bounds
X_e2 = base_e2[campos_e2].to_numpy(dtype=float)
y_e2 = base_e2["classe"].to_numpy()
classe_arvore_e2 = "arvore" if "arvore" in set(y_e2) else None

for tamanho_e2 in tam_blocos_e2:
    col_e2 = np.floor((centros_e2.x.to_numpy() - xmin_e2) / tamanho_e2).astype(int)
    lin_e2 = np.floor((centros_e2.y.to_numpy() - ymin_e2) / tamanho_e2).astype(int)
    grupos_e2 = np.array([f"{x}_{y}" for x, y in zip(col_e2, lin_e2)])
    total_blocos_e2 = len(np.unique(grupos_e2))
    if total_blocos_e2 < 5:
        ignorados_e2.append((tamanho_e2, 0, "menos de cinco blocos distintos"))
        continue

    splitter_e2 = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    for rodada_e2, (tr_e2, te_e2) in enumerate(
        splitter_e2.split(X_e2, y_e2, groups=grupos_e2), start=1
    ):
        assert not (set(grupos_e2[tr_e2]) & set(grupos_e2[te_e2]))
        assert not (
            set(base_e2.iloc[tr_e2]["segmento_id"])
            & set(base_e2.iloc[te_e2]["segmento_id"])
        )
        # Excluir do treino objetos a até 25m das geometrias do teste.
        faixa_e2 = base_e2.geometry.iloc[te_e2].buffer(faixa_exclusao_e2)
        area_teste_e2 = (faixa_e2.union_all() if hasattr(faixa_e2, "union_all")
                         else faixa_e2.unary_union)
        manter_e2 = ~base_e2.geometry.iloc[tr_e2].intersects(area_teste_e2).to_numpy()
        tr_seguro_e2 = tr_e2[manter_e2]
        if len(tr_seguro_e2) < 15 or len(te_e2) < 5:
            ignorados_e2.append((tamanho_e2, rodada_e2, "amostra insuficiente após exclusão"))
            continue
        if (len(np.unique(y_e2[tr_seguro_e2])) < 2
                or len(np.unique(y_e2[te_e2])) < 2):
            ignorados_e2.append((tamanho_e2, rodada_e2, "uma classe ausente em treino ou teste"))
            continue

        rf_e2 = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
        rf_e2.fit(X_e2[tr_seguro_e2], y_e2[tr_seguro_e2])
        previsto_e2 = rf_e2.predict(X_e2[te_e2])
        linha_e2 = {
            "bloco_m": tamanho_e2, "rodada": rodada_e2,
            "blocos_total": total_blocos_e2,
            "blocos_teste": len(np.unique(grupos_e2[te_e2])),
            "n_treino": len(tr_seguro_e2), "n_teste": len(te_e2),
            "n_excluidos_faixa": len(tr_e2) - len(tr_seguro_e2),
            "teste_arvore": int((y_e2[te_e2] == "arvore").sum()),
            "teste_grama": int((y_e2[te_e2] == "grama").sum()),
            "acuracia": accuracy_score(y_e2[te_e2], previsto_e2),
            "acuracia_balanceada": balanced_accuracy_score(y_e2[te_e2], previsto_e2),
            "f1_macro": f1_score(y_e2[te_e2], previsto_e2, average="macro", zero_division=0),
        }
        if classe_arvore_e2 is not None:
            linha_e2["precisao_arvore"] = precision_score(
                y_e2[te_e2], previsto_e2, pos_label="arvore", zero_division=0
            )
            linha_e2["recall_arvore"] = recall_score(
                y_e2[te_e2], previsto_e2, pos_label="arvore", zero_division=0
            )
            linha_e2["f1_arvore"] = f1_score(
                y_e2[te_e2], previsto_e2, pos_label="arvore", zero_division=0
            )
        resultados_e2.append(linha_e2)

tabela_e2 = pd.DataFrame(resultados_e2)
print("=== E2 | Resultados por separação geográfica ===")
if tabela_e2.empty:
    print("Nenhuma rodada foi viável com os blocos, as classes e a faixa definidos.")
else:
    print(tabela_e2.round(3).to_string(index=False))
    print("\nResumo descritivo por tamanho de bloco:")
    print(tabela_e2.groupby("bloco_m")[["acuracia", "acuracia_balanceada", "f1_macro"]]
          .agg(["count", "mean", "std"]).round(3).to_string())
print("\nRodadas ignoradas:", len(ignorados_e2))
for tamanho_e2, rodada_e2, motivo_e2 in ignorados_e2:
    print(f"  Bloco {tamanho_e2}m; rodada {rodada_e2}: {motivo_e2}")
print("\nINTERPRETAÇÃO: repetição de divisões na mesma área NÃO equivale a")
print("testes em novas cidades; desvios-padrão entre rodadas não são intervalos")
print("de confiança independentes nem comprovam ganho sobre a acurácia 0,87 histórica.")


## E3 — comparação pareada de classificadores (somente Area1)

**Objetivo:** comparar os resultados de **Random Forest (RF)**, **SVM de kernel RBF**, **Regressão Logística** e uma referência ingênua que sempre prevê a classe majoritária (**DummyClassifier**), utilizando **exatamente os mesmos registros de teste em cada rodada**.

- Reutiliza os segmentos únicos e as três características `mean_intensity` (brilho HSV), `mean_edges` (bordas Canny) e `mean_ndvi` (NDVI), preparados em **E2a**.
- Reproduz o protocolo do **E2b**: blocos de 250 e 500 m, 10 tentativas por escala, semente 42, 20% dos **blocos** reservados para teste e exclusão de objetos de treino a até 25 m da geometria de teste.
- SVM e regressão logística aplicam `StandardScaler` **ajustado exclusivamente no treino**; o RF usa os mesmos 500 estimadores do experimento original.
- Não há busca de hiperparâmetros no conjunto de teste. Os quatro métodos utilizam as mesmas entradas e divisão; são classificadores de **árvore versus gramínea**, não detectores completos comparáveis a YOLO sem anotações adicionais.
- Repetições de teste compartilham amostras; as diferenças pareadas são descritivas, **não** testes independentes de significância.
- A análise continua restrita à **mesma área Area1**; os dados atuais não permitem alegar generalização externa, validar altura real ou interferência nos cabos.

**Execução:** rode as células originais e **E2a**, depois as duas células E3 a seguir. A execução de **E2b** não é obrigatória, mas, se tiver sido concluída, o E3 também confere a consistência de suas métricas do RF. Nenhuma métrica nova é presumida antes de executar o Colab.


In [ ]:
# E3a — RF vs SVM vs regressão logística vs baseline, nas MESMAS partições espaciais de E2
# Pré-requisito: E2a concluída; não altera rf_classifier, df ou os dados originais.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
)
import numpy as np
import pandas as pd

if "base_e2" not in globals() or "campos_e2" not in globals():
    raise RuntimeError("Execute E2a para criar base_e2/campos_e2 antes de E3.")
assert base_e2.crs is not None and base_e2.crs.is_projected, (
    "A geometria de E3 deve usar um CRS projetado com unidades em metros."
)
assert len(base_e2) == base_e2["segmento_id"].nunique(), (
    "Há segmentos repetidos; verifique a preparação E2a."
)
assert set(base_e2["classe"].unique()) == {"arvore", "grama"}, (
    "Esperadas as classes 'arvore' e 'grama' em E3."
)

X_e3 = base_e2[campos_e2].to_numpy(dtype=float)
y_e3 = base_e2["classe"].to_numpy()
if not np.isfinite(X_e3).all():
    raise ValueError("Atributos não finitos em E3.")
centros_e3 = base_e2.geometry.centroid
xmin_e3, ymin_e3, _, _ = base_e2.total_bounds
resultados_e3, ignorados_e3 = [], []
protocolos_validos_e3 = []

for tamanho_e3 in [250, 500]:
    col_e3 = np.floor((centros_e3.x.to_numpy() - xmin_e3) / tamanho_e3).astype(int)
    lin_e3 = np.floor((centros_e3.y.to_numpy() - ymin_e3) / tamanho_e3).astype(int)
    grupos_e3 = np.array([f"{c}_{l}" for c, l in zip(col_e3, lin_e3)])
    n_blocos_e3 = len(np.unique(grupos_e3))
    if n_blocos_e3 < 5:
        ignorados_e3.append((tamanho_e3, 0, "menos de cinco blocos"))
        continue

    separador_e3 = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    for rodada_e3, (treino_e3, teste_e3) in enumerate(
        separador_e3.split(X_e3, y_e3, groups=grupos_e3), start=1
    ):
        if set(grupos_e3[treino_e3]) & set(grupos_e3[teste_e3]):
            raise AssertionError("Vazamento de blocos geográficos entre treino e teste.")
        if (set(base_e2.iloc[treino_e3]["segmento_id"]) &
                set(base_e2.iloc[teste_e3]["segmento_id"])):
            raise AssertionError("Mesmo segmento em treino e teste.")
        # Mesmo embargo espacial aplicado em E2b.
        buffers_e3 = base_e2.geometry.iloc[teste_e3].buffer(25)
        zona_e3 = (buffers_e3.union_all() if hasattr(buffers_e3, "union_all")
                   else buffers_e3.unary_union)
        afastados_e3 = ~base_e2.geometry.iloc[treino_e3].intersects(zona_e3).to_numpy()
        treino_final_e3 = treino_e3[afastados_e3]

        if len(treino_final_e3) < 15 or len(teste_e3) < 5:
            ignorados_e3.append((tamanho_e3, rodada_e3, "amostra insuficiente"))
            continue
        if (len(np.unique(y_e3[treino_final_e3])) != 2
                or len(np.unique(y_e3[teste_e3])) != 2):
            ignorados_e3.append((tamanho_e3, rodada_e3, "faltou uma classe"))
            continue

        modelos_e3 = {
            "Dummy (maioria)": DummyClassifier(strategy="most_frequent"),
            "Regressão logística": make_pipeline(
                StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)
            ),
            "SVM (RBF)": make_pipeline(
                StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")
            ),
            "Random Forest": RandomForestClassifier(
                n_estimators=500, random_state=42, n_jobs=-1
            ),
        }
        protocolos_validos_e3.append((tamanho_e3, rodada_e3))
        for nome_e3, modelo_e3 in modelos_e3.items():
            modelo_e3.fit(X_e3[treino_final_e3], y_e3[treino_final_e3])
            previsoes_e3 = modelo_e3.predict(X_e3[teste_e3])
            resultados_e3.append({
                "bloco_m": tamanho_e3, "rodada": rodada_e3, "modelo": nome_e3,
                "n_treino": len(treino_final_e3), "n_teste": len(teste_e3),
                "n_excluidos_faixa": len(treino_e3) - len(treino_final_e3),
                "teste_arvore": int(np.sum(y_e3[teste_e3] == "arvore")),
                "teste_grama": int(np.sum(y_e3[teste_e3] == "grama")),
                "acuracia": accuracy_score(y_e3[teste_e3], previsoes_e3),
                "acuracia_balanceada": balanced_accuracy_score(
                    y_e3[teste_e3], previsoes_e3
                ),
                "f1_macro": f1_score(
                    y_e3[teste_e3], previsoes_e3, average="macro", zero_division=0
                ),
                "precisao_arvore": precision_score(
                    y_e3[teste_e3], previsoes_e3, pos_label="arvore", zero_division=0
                ),
                "recall_arvore": recall_score(
                    y_e3[teste_e3], previsoes_e3, pos_label="arvore", zero_division=0
                ),
                "f1_arvore": f1_score(
                    y_e3[teste_e3], previsoes_e3, pos_label="arvore", zero_division=0
                ),
            })

tabela_e3 = pd.DataFrame(resultados_e3)
print("=== E3 | Protocolo pareado entre classificadores ===")
print("Partições espaciais válidas:", len(protocolos_validos_e3))
print("Partições ignoradas:", len(ignorados_e3))
for b_e3, r_e3, motivo_e3 in ignorados_e3:
    print(f"  {b_e3} m | rodada {r_e3}: {motivo_e3}")

if tabela_e3.empty:
    print("Nenhuma partição permitiu comparar os modelos; não há métricas E3.")
else:
    n_modelos_e3 = tabela_e3.groupby(["bloco_m", "rodada"])["modelo"].nunique()
    assert (n_modelos_e3 == 4).all(), "Algum modelo não participou da mesma partição."
    print("\nMétricas por partição e algoritmo:")
    print(tabela_e3.round(3).to_string(index=False))
    print("\nResumo por escala e algoritmo — média e desvio-padrão descritivos:")
    resumo_e3 = tabela_e3.groupby(["bloco_m", "modelo"])[
        ["acuracia", "acuracia_balanceada", "f1_macro", "f1_arvore"]
    ].agg(["count", "mean", "std"]).round(3)
    print(resumo_e3.to_string())

    # Se E2b foi executado, suas métricas do RF devem coincidir na mesma partição.
    if ("tabela_e2" in globals() and isinstance(tabela_e2, pd.DataFrame)
            and not tabela_e2.empty):
        rf_e3 = tabela_e3.loc[tabela_e3["modelo"] == "Random Forest"]
        verif_e3 = rf_e3.merge(
            tabela_e2[["bloco_m", "rodada", "n_treino", "n_teste", "f1_macro"]],
            on=["bloco_m", "rodada"], how="outer",
            suffixes=("_e3", "_e2"), indicator=True
        )
        if not (verif_e3["_merge"] == "both").all():
            raise AssertionError("E2 e E3 não usaram as mesmas partições válidas.")
        assert np.array_equal(verif_e3["n_treino_e3"], verif_e3["n_treino_e2"])
        assert np.array_equal(verif_e3["n_teste_e3"], verif_e3["n_teste_e2"])
        assert np.allclose(verif_e3["f1_macro_e3"], verif_e3["f1_macro_e2"])
        print("\nConferência E2 × E3: mesmas divisões e métricas RF confirmadas.")

print("\nIMPORTANTE: não comparar diretamente estes valores com a acurácia 0,87")
print("histórica, calculada sob protocolo diferente.")


In [ ]:
# E3b — diferenças PAREADAS (mesmo teste por rodada); tabela e gráfico exploratório
# Não interpreta rodadas repetidas como observações estatisticamente independentes.
import matplotlib.pyplot as plt

if tabela_e3.empty:
    print("Sem métricas E3 para resumir ou representar graficamente.")
else:
    f1_pares_e3 = tabela_e3.pivot(
        index=["bloco_m", "rodada"], columns="modelo", values="f1_macro"
    )
    bal_pares_e3 = tabela_e3.pivot(
        index=["bloco_m", "rodada"], columns="modelo",
        values="acuracia_balanceada"
    )
    for alternativa_e3 in ["SVM (RBF)", "Regressão logística", "Dummy (maioria)"]:
        diferenca_e3 = f1_pares_e3["Random Forest"] - f1_pares_e3[alternativa_e3]
        print(f"\nDiferença de F1-macro por rodada: RF − {alternativa_e3}")
        print(diferenca_e3.groupby(level="bloco_m").agg(["count", "mean", "std"])
              .round(3).to_string())
    print("\nAcurácia balanceada: RF e concorrentes (mesmos testes):")
    print(bal_pares_e3.groupby(level="bloco_m").mean().round(3).to_string())

    for bloco_graf_e3, recorte_e3 in tabela_e3.groupby("bloco_m"):
        ordem_e3 = ["Dummy (maioria)", "Regressão logística", "SVM (RBF)",
                     "Random Forest"]
        media_e3 = recorte_e3.groupby("modelo")["f1_macro"].mean().reindex(ordem_e3)
        plt.figure(figsize=(9, 4))
        plt.bar(media_e3.index, media_e3.to_numpy())
        plt.ylim(0, 1)
        plt.ylabel("F1-macro médio nas partições válidas")
        plt.title(f"E3 — classificação com blocos espaciais de {bloco_graf_e3} m")
        plt.xticks(rotation=18, ha="right")
        plt.tight_layout()
        plt.show()

    print("\nConclusões dependem dos valores reais calculados após executar no Colab.")
    print("A comparação mede classificação árvore/grama na Area1 e não risco elétrico.")


## E3c — bateria adicional: oito algoritmos, escalas e embargo espacial

**Justificativa:** reforçar a comparação exigida pelo orientador com **RF, SVM RBF, regressão logística, Dummy, Extra Trees, KNN, HistGradientBoosting e Naive Bayes** sobre os **mesmos segmentos rotulados originais**. Comparar **blocos 250/500 m**, **embargos 0/25/50 m**, **cinco divisões por escala** (até 30 protocolos e 240 ajustes de modelo, apenas se todos forem válidos). São análises de sensibilidade e não 240 amostras independentes.

**Critérios pré-especificados:** F1-macro primário; acurácia balanceada, F1/precisão/recall da classe árvore e MCC secundários. Treino/teste compartilham o mesmo protocolo por algoritmo, sem vazamento de blocos/segmentos. A normalização está dentro de cada pipeline, ajustada **somente no treino**. Não otimizar hiperparâmetros olhando para o teste. A análise **não** incorpora novos rótulos, imagens ou dados externos.

**Limitações:** conjuntos de teste podem se repetir entre divisões GroupShuffleSplit, a AOI é única e o número de amostras rotuladas é pequeno. Médias/desvios-padrão e diferenças pareadas são **descritivos**, sem p-valor ou afirmação de superioridade estatística. Comparações com o 0,87 histórico não são válidas.

In [ ]:
# E3c — oito classificadores, três embargos e duas escalas; somente Area1
# Requer E2a; não sobrescreve classificadores, variáveis ou tabelas históricas.
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, matthews_corrcoef
)

if "base_e2" not in globals() or "campos_e2" not in globals():
    raise RuntimeError("Execute E2a antes da bateria E3c.")
if base_e2.crs is None or not base_e2.crs.is_projected:
    raise ValueError("E3c requer CRS projetado em metros.")
if len(base_e2) != base_e2["segmento_id"].nunique():
    raise ValueError("E3c: há segmentos duplicados.")
if set(base_e2["classe"].unique()) != {"arvore", "grama"}:
    raise ValueError("E3c: esperadas apenas as classes arvore/grama.")

X_e3c = base_e2[campos_e2].to_numpy(dtype=float)
y_e3c = base_e2["classe"].to_numpy()
if not np.isfinite(X_e3c).all():
    raise ValueError("E3c: atributos com NaN ou infinito.")
centroides_e3c = base_e2.geometry.centroid
x0_e3c, y0_e3c, _, _ = base_e2.total_bounds
resultados_e3c, ignorados_e3c = [], []

def modelos_extras_e3c():
    return {
        "Dummy (maioria)": DummyClassifier(strategy="most_frequent"),
        "Regressão logística": make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)),
        "SVM (RBF)": make_pipeline(StandardScaler(), SVC(C=1, kernel="rbf", gamma="scale")),
        "Random Forest": RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
        "Extra Trees": ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        "KNN (7)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=100, learning_rate=0.1, l2_regularization=1.0, random_state=42),
        "Gaussian Naive Bayes": GaussianNB(),
    }

for bloco_e3c in (250, 500):
    gx_e3c = np.floor((centroides_e3c.x.to_numpy() - x0_e3c) / bloco_e3c).astype(int)
    gy_e3c = np.floor((centroides_e3c.y.to_numpy() - y0_e3c) / bloco_e3c).astype(int)
    grupos_e3c = np.array([f"{a}_{b}" for a, b in zip(gx_e3c, gy_e3c)])
    if len(np.unique(grupos_e3c)) < 5:
        ignorados_e3c.append((bloco_e3c, 0, -1, "menos de cinco blocos"))
        continue
    divisao_e3c = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=2026)
    for rodada_e3c, (tr_bruto_e3c, te_e3c) in enumerate(
        divisao_e3c.split(X_e3c, y_e3c, groups=grupos_e3c), start=1
    ):
        if set(grupos_e3c[tr_bruto_e3c]) & set(grupos_e3c[te_e3c]):
            raise AssertionError("Vazamento geográfico E3c.")
        if (set(base_e2.iloc[tr_bruto_e3c]["segmento_id"])
                & set(base_e2.iloc[te_e3c]["segmento_id"])):
            raise AssertionError("Vazamento de identificadores E3c.")
        for embargo_e3c in (0, 25, 50):
            if embargo_e3c == 0:
                tr_e3c = tr_bruto_e3c.copy()
            else:
                geoms_e3c = base_e2.geometry.iloc[te_e3c].buffer(embargo_e3c)
                zona_e3c = (geoms_e3c.union_all() if hasattr(geoms_e3c, "union_all")
                           else geoms_e3c.unary_union)
                mascara_e3c = ~base_e2.geometry.iloc[tr_bruto_e3c].intersects(
                    zona_e3c).to_numpy()
                tr_e3c = tr_bruto_e3c[mascara_e3c]
            if (len(tr_e3c) < 15 or len(te_e3c) < 7
                    or len(np.unique(y_e3c[tr_e3c])) != 2
                    or len(np.unique(y_e3c[te_e3c])) != 2):
                ignorados_e3c.append(
                    (bloco_e3c, rodada_e3c, embargo_e3c, "tamanho/classes insuficientes"))
                continue
            if embargo_e3c:
                if base_e2.geometry.iloc[tr_e3c].intersects(zona_e3c).any():
                    raise AssertionError("Embargo espacial não aplicado.")
            for nome_e3c, clf_e3c in modelos_extras_e3c().items():
                clf_e3c.fit(X_e3c[tr_e3c], y_e3c[tr_e3c])
                previsto_e3c = clf_e3c.predict(X_e3c[te_e3c])
                resultados_e3c.append({
                    "bloco_m": bloco_e3c, "rodada": rodada_e3c,
                    "embargo_m": embargo_e3c, "modelo": nome_e3c,
                    "n_treino": int(len(tr_e3c)), "n_teste": int(len(te_e3c)),
                    "teste_arvore": int((y_e3c[te_e3c] == "arvore").sum()),
                    "teste_grama": int((y_e3c[te_e3c] == "grama").sum()),
                    "f1_macro": f1_score(y_e3c[te_e3c], previsto_e3c,
                                         average="macro", zero_division=0),
                    "acuracia": accuracy_score(y_e3c[te_e3c], previsto_e3c),
                    "acuracia_balanceada": balanced_accuracy_score(
                        y_e3c[te_e3c], previsto_e3c),
                    "f1_arvore": f1_score(y_e3c[te_e3c], previsto_e3c,
                                          pos_label="arvore", zero_division=0),
                    "precisao_arvore": precision_score(y_e3c[te_e3c], previsto_e3c,
                                                       pos_label="arvore", zero_division=0),
                    "recall_arvore": recall_score(y_e3c[te_e3c], previsto_e3c,
                                                  pos_label="arvore", zero_division=0),
                    "mcc": matthews_corrcoef(y_e3c[te_e3c], previsto_e3c),
                })

tabela_e3c = pd.DataFrame(resultados_e3c)
tabela_e3c_pareada = pd.DataFrame()
if tabela_e3c.empty:
    print("E3c: nenhuma partição válida; não há resultados de comparação.")
else:
    n_e3c = tabela_e3c.groupby(["bloco_m", "rodada", "embargo_m"])["modelo"].nunique()
    assert (n_e3c == 8).all(), "Os algoritmos não compartilharam todas as divisões."
    print("E3c — ajustes efetivamente realizados:", len(tabela_e3c))
    print("E3c — protocolos válidos:", len(n_e3c), "| ignorados:", len(ignorados_e3c))
    print(tabela_e3c.groupby(["bloco_m", "embargo_m", "modelo"])[
        ["f1_macro", "acuracia_balanceada", "mcc"]].agg(["count", "mean", "std"])
        .round(4).to_string())
    pivot_e3c = tabela_e3c.pivot(
        index=["bloco_m", "rodada", "embargo_m"], columns="modelo", values="f1_macro")
    tabela_e3c_pareada = (
        pivot_e3c.sub(pivot_e3c["Random Forest"], axis=0)
        .drop(columns=["Random Forest"]).stack().rename("delta_f1_vs_rf")
        .reset_index().rename(columns={"modelo": "alternativa"})
    )
    print("\nDiferenças pareadas (alternativa − Random Forest):")
    print(tabela_e3c_pareada.groupby(["bloco_m", "embargo_m", "alternativa"])[
        "delta_f1_vs_rf"].agg(["count", "mean", "std"]).round(4).to_string())
    print("\nIMPORTANTE: divisões repetidas dentro da mesma AOI não são réplicas independentes.")
print("E3c — partições ignoradas:", ignorados_e3c)

## E4 — ablação de atributos: NDVI, brilho HSV e bordas Canny

**Pergunta científica:** quais dos três atributos originais contribuem para diferenciar árvores de gramíneas, e quais combinações permitem o melhor desempenho?

São avaliadas as **sete combinações não vazias** de:
- `mean_intensity` = brilho HSV;
- `mean_edges` = bordas Canny;
- `mean_ndvi` = NDVI.

A comparação é feita com **Random Forest (500 árvores), SVM RBF e regressão logística**. O **Dummy majoritário** é incluído como linha de referência — ele ignora características e, por isso, tem a mesma previsão nas sete combinações. Em todas as avaliações, os modelos compartilham **os mesmos blocos geográficos de teste**, a mesma deduplicação de objetos e a mesma faixa de exclusão de 25 metros já definidas no E2/E3. A padronização é ajustada só com o conjunto de treino. Não são utilizados novos dados.

**Como executar:** termine as células originais e execute **E2a**, que constrói `base_e2`; depois execute E4a e E4b. As células E3 não são obrigatórias para E4, mas, se executadas, permitem comparar o protocolo espacial completo com E3.

**Interpretação:** os experimentos medem classificação árvore/grama na área `Area1`, não aferem altura real nem interferência em cabos. Rodadas repetidas podem compartilhar dados; seus desvios-padrão são descritivos. Nenhum número novo de desempenho é afirmado antes de executar o Colab.


In [ ]:
# E4a — ablação das 7 combinações, com partições ESPACIAIS idênticas a E2/E3
# Requer base_e2/campos_e2 da célula E2a. Não muda modelos/saídas originais.
import itertools
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score

if "base_e2" not in globals() or "campos_e2" not in globals():
    raise RuntimeError("Execute E2a antes de E4: base_e2/campos_e2 ausentes.")
features_e4 = ["mean_intensity", "mean_edges", "mean_ndvi"]
nomes_features_e4 = {
    "mean_intensity": "Brilho HSV", "mean_edges": "Bordas Canny",
    "mean_ndvi": "NDVI"
}
assert all(f in base_e2.columns for f in features_e4)
assert base_e2.crs is not None and base_e2.crs.is_projected, "CRS deve ser métrico."
assert len(base_e2) == base_e2["segmento_id"].nunique(), "Segmentos duplicados em E2a."
assert set(base_e2["classe"].unique()) == {"arvore", "grama"}

X_total_e4 = base_e2[features_e4].to_numpy(dtype=float)
y_e4 = base_e2["classe"].to_numpy()
assert np.isfinite(X_total_e4).all(), "Atributos não finitos."
combinacoes_e4 = [
    tuple(combo)
    for k in range(1, len(features_e4) + 1)
    for combo in itertools.combinations(features_e4, k)
]
assert len(combinacoes_e4) == 7
centros_e4 = base_e2.geometry.centroid
xmin_e4, ymin_e4, _, _ = base_e2.total_bounds
linhas_e4, excluidos_e4 = [], []
particoes_validas_e4 = []

for tamanho_e4 in [250, 500]:
    col_e4 = np.floor((centros_e4.x.to_numpy() - xmin_e4) / tamanho_e4).astype(int)
    lin_e4 = np.floor((centros_e4.y.to_numpy() - ymin_e4) / tamanho_e4).astype(int)
    grupos_e4 = np.array([f"{c}_{l}" for c, l in zip(col_e4, lin_e4)])
    blocos_e4 = len(np.unique(grupos_e4))
    if blocos_e4 < 5:
        excluidos_e4.append((tamanho_e4, 0, "menos de cinco blocos"))
        continue

    partidor_e4 = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    for rodada_e4, (tr_e4, te_e4) in enumerate(
        partidor_e4.split(X_total_e4, y_e4, groups=grupos_e4), start=1
    ):
        assert not (set(grupos_e4[tr_e4]) & set(grupos_e4[te_e4]))
        assert not (
            set(base_e2.iloc[tr_e4]["segmento_id"]) &
            set(base_e2.iloc[te_e4]["segmento_id"])
        ), "Vazamento de segmentos entre treino e teste."
        buffer_e4 = base_e2.geometry.iloc[te_e4].buffer(25)
        geometria_teste_e4 = (buffer_e4.union_all() if hasattr(buffer_e4, "union_all")
                              else buffer_e4.unary_union)
        afastados_e4 = ~base_e2.geometry.iloc[tr_e4].intersects(
            geometria_teste_e4
        ).to_numpy()
        tr_seguro_e4 = tr_e4[afastados_e4]
        if len(tr_seguro_e4) < 15 or len(te_e4) < 5:
            excluidos_e4.append((tamanho_e4, rodada_e4, "amostras insuficientes"))
            continue
        if (len(np.unique(y_e4[tr_seguro_e4])) < 2
                or len(np.unique(y_e4[te_e4])) < 2):
            excluidos_e4.append((tamanho_e4, rodada_e4, "classe ausente"))
            continue

        particoes_validas_e4.append((tamanho_e4, rodada_e4))
        # Dummy: apenas UMA predição por partição, pois não utiliza atributos.
        referencia_e4 = DummyClassifier(strategy="most_frequent")
        referencia_e4.fit(X_total_e4[tr_seguro_e4], y_e4[tr_seguro_e4])
        previsao_dummy_e4 = referencia_e4.predict(X_total_e4[te_e4])

        for combo_e4 in combinacoes_e4:
            idx_features_e4 = [features_e4.index(f) for f in combo_e4]
            X_combo_e4 = X_total_e4[:, idx_features_e4]
            etiqueta_e4 = " + ".join(nomes_features_e4[f] for f in combo_e4)
            modelos_e4 = {
                "Random Forest": RandomForestClassifier(
                    n_estimators=500, random_state=42, n_jobs=-1
                ),
                "SVM (RBF)": make_pipeline(
                    StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")
                ),
                "Regressão logística": make_pipeline(
                    StandardScaler(),
                    LogisticRegression(max_iter=2000, random_state=42)
                ),
            }
            for nome_e4, modelo_e4 in modelos_e4.items():
                modelo_e4.fit(X_combo_e4[tr_seguro_e4], y_e4[tr_seguro_e4])
                previsao_e4 = modelo_e4.predict(X_combo_e4[te_e4])
                linhas_e4.append({
                    "bloco_m": tamanho_e4, "rodada": rodada_e4, "modelo": nome_e4,
                    "atributos": etiqueta_e4, "quantidade_atributos": len(combo_e4),
                    "variaveis": "|".join(combo_e4),
                    "n_treino": len(tr_seguro_e4), "n_teste": len(te_e4),
                    "acuracia": accuracy_score(y_e4[te_e4], previsao_e4),
                    "acuracia_balanceada": balanced_accuracy_score(
                        y_e4[te_e4], previsao_e4
                    ),
                    "f1_macro": f1_score(
                        y_e4[te_e4], previsao_e4, average="macro", zero_division=0
                    ),
                    "f1_arvore": f1_score(
                        y_e4[te_e4], previsao_e4, pos_label="arvore", zero_division=0
                    ),
                    "recall_arvore": recall_score(
                        y_e4[te_e4], previsao_e4, pos_label="arvore", zero_division=0
                    ),
                })
            # A referência Dummy é a mesma em todas as ablações desta partição.
            linhas_e4.append({
                "bloco_m": tamanho_e4, "rodada": rodada_e4, "modelo": "Dummy (maioria)",
                "atributos": etiqueta_e4, "quantidade_atributos": len(combo_e4),
                "variaveis": "|".join(combo_e4),
                "n_treino": len(tr_seguro_e4), "n_teste": len(te_e4),
                "acuracia": accuracy_score(y_e4[te_e4], previsao_dummy_e4),
                "acuracia_balanceada": balanced_accuracy_score(
                    y_e4[te_e4], previsao_dummy_e4
                ),
                "f1_macro": f1_score(
                    y_e4[te_e4], previsao_dummy_e4, average="macro", zero_division=0
                ),
                "f1_arvore": f1_score(
                    y_e4[te_e4], previsao_dummy_e4, pos_label="arvore", zero_division=0
                ),
                "recall_arvore": recall_score(
                    y_e4[te_e4], previsao_dummy_e4, pos_label="arvore", zero_division=0
                ),
            })

tabela_e4 = pd.DataFrame(linhas_e4)
print("=== E4 | Ablação dos atributos com Area1 ===")
print("Combinações:", len(combinacoes_e4), "| Modelos: 4 (3 supervisionados + Dummy)")
print("Partições espaciais válidas:", len(particoes_validas_e4))
print("Partições descartadas:", len(excluidos_e4))
for tam_e4, num_e4, causa_e4 in excluidos_e4:
    print(f"  {tam_e4} m / rodada {num_e4}: {causa_e4}")
if tabela_e4.empty:
    print("Sem partições válidas — nenhum desempenho novo foi produzido.")
else:
    totais_e4 = tabela_e4.groupby(["bloco_m", "rodada", "modelo"]).size()
    assert (totais_e4 == 7).all(), "Alguma ablação está ausente numa partição."
    print("\nResumo descritivo — F1-macro por escala / modelo / atributos:")
    print(
        tabela_e4.groupby(["bloco_m", "modelo", "atributos"])[
            ["f1_macro", "acuracia_balanceada", "recall_arvore"]
        ].mean().round(3).to_string()
    )


In [ ]:
# E4b — comparação PAREADA de cada subconjunto com as três características originais
# O mesmo conjunto de teste é utilizado em todas as linhas de cada partição.
import matplotlib.pyplot as plt

if tabela_e4.empty:
    print("E4 sem rodadas válidas; não há comparações nem gráficos.")
else:
    full_key_e4 = "|".join(features_e4)
    colunas_chave_e4 = ["bloco_m", "rodada", "modelo"]
    full_e4 = tabela_e4.loc[
        tabela_e4["variaveis"] == full_key_e4,
        colunas_chave_e4 + ["f1_macro", "acuracia_balanceada"]
    ].rename(columns={
        "f1_macro": "f1_full", "acuracia_balanceada": "balanceada_full"
    })
    comparacoes_e4 = tabela_e4.merge(
        full_e4, on=colunas_chave_e4, how="left", validate="many_to_one"
    )
    assert comparacoes_e4["f1_full"].notna().all()
    comparacoes_e4["delta_f1_vs_full"] = (
        comparacoes_e4["f1_macro"] - comparacoes_e4["f1_full"]
    )
    print("=== E4 | Diferença pareada de F1-macro: subconjunto MENOS os 3 atributos ===")
    supervis_e4 = comparacoes_e4.loc[
        comparacoes_e4["modelo"] != "Dummy (maioria)"
    ]
    deltas_e4 = supervis_e4.groupby(
        ["bloco_m", "modelo", "atributos"]
    )["delta_f1_vs_full"].agg(["count", "mean", "std"]).round(3)
    print(deltas_e4.to_string())

    # Análise de retirada individual: o efeito é observacional e depende da amostra.
    print("\n=== E4 | Retirada individual de atributos (F1 full MENOS F1 sem atributo) ===")
    for removida_e4 in features_e4:
        faltando_e4 = "|".join(f for f in features_e4 if f != removida_e4)
        analise_e4 = supervis_e4.loc[
            supervis_e4["variaveis"] == faltando_e4
        ].copy()
        analise_e4["perda_ao_remover"] = -analise_e4["delta_f1_vs_full"]
        print("\nExclusão:", nomes_features_e4[removida_e4])
        print(analise_e4.groupby(["bloco_m", "modelo"])["perda_ao_remover"]
              .agg(["count", "mean", "std"]).round(3).to_string())

    for tamanho_graf_e4 in [250, 500]:
        subconjunto_graf_e4 = supervis_e4.loc[
            supervis_e4["bloco_m"] == tamanho_graf_e4
        ]
        if subconjunto_graf_e4.empty:
            continue
        colunas_e4 = [
            "Brilho HSV", "Bordas Canny", "NDVI",
            "Brilho HSV + Bordas Canny", "Brilho HSV + NDVI",
            "Bordas Canny + NDVI", "Brilho HSV + Bordas Canny + NDVI"
        ]
        medias_e4 = subconjunto_graf_e4.pivot_table(
            index="atributos", columns="modelo", values="f1_macro", aggfunc="mean"
        ).reindex(colunas_e4)
        fig, ax = plt.subplots(figsize=(12, 5))
        medias_e4.plot(kind="bar", ax=ax)
        ax.set_title(f"E4 — F1-macro médio por atributos e modelo ({tamanho_graf_e4} m)")
        ax.set_xlabel("Combinação de características")
        ax.set_ylabel("F1-macro (média descritiva)")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=35)
        fig.tight_layout()
        plt.show()

    # Checagem contra os métodos do E3, se E3 também foi executado na sessão.
    if ("tabela_e3" in globals() and isinstance(tabela_e3, pd.DataFrame)
            and not tabela_e3.empty):
        full_e4_check = tabela_e4.loc[
            tabela_e4["variaveis"] == full_key_e4,
            ["bloco_m", "rodada", "modelo", "n_treino", "n_teste", "f1_macro"]
        ]
        comparavel_e3 = tabela_e3[
            ["bloco_m", "rodada", "modelo", "n_treino", "n_teste", "f1_macro"]
        ]
        teste_e3e4 = full_e4_check.merge(
            comparavel_e3,
            on=["bloco_m", "rodada", "modelo"],
            suffixes=("_e4", "_e3"),
            how="outer", indicator=True, validate="one_to_one"
        )
        assert (teste_e3e4["_merge"] == "both").all(), (
            "As partições/modelos de E3 e E4 não são idênticos."
        )
        for metrica_e4 in ["n_treino", "n_teste", "f1_macro"]:
            assert np.allclose(
                teste_e3e4[f"{metrica_e4}_e3"],
                teste_e3e4[f"{metrica_e4}_e4"]
            ), f"E3 difere de E4 com os 3 atributos: {metrica_e4}"
        print("\nChecagem E3 × E4: métricas idênticas com os três atributos.")

    print("\nATENÇÃO: ganhos ou perdas aqui são descritivos na mesma Area1.")
    print("Não inferir superioridade geral nem importância causal sem validação externa.")


## E5 — sensibilidade do Quickshift e do K-Means (dados da Area1)

**Objetivo:** medir como mudanças de parâmetros afetam a segmentação e o agrupamento, sem modificar as 116 células históricas nem exigir novos dados.

São **dois ensaios distintos**, pois alterar o Quickshift muda a identidade e a geometria dos segmentos; não seria válido reutilizar, sem refazer a associação dos rótulos e a extração de atributos, as mesmas linhas de treino/teste de E2–E4.

- **E5a — Quickshift:** aplica 5 configurações em **quatro janelas fixas de 256 × 256 pixels**, escolhidas deterministicamente perto das áreas de referência para árvores e gramíneas já existentes. Mede tempo de execução, número de segmentos, mediana da área de segmentos e fração de segmentos muito pequenos. A configuração original é `kernel_size=3, max_dist=5, ratio=0.04`. **Os resultados das janelas não são equivalentes aos do raster inteiro.**
- **E5b — K-Means:** sobre **uma amostra fixa de até 1.500 polígonos** da segmentação original (priorizando amostras com rótulo e incluindo amostra do restante), extrai das mesmas bandas RGB/NIR as oito características originais (médias e desvios-padrão). Testa **K = 4, 6, 8** com `random_state=42, n_init=10`, sem padronização, como no código original. Mede inércia, silhouette (amostral), tempo, estabilidade de grupos por **Adjusted Rand Index (ARI)** e distribuição de classes rotuladas nos grupos.
- **Memória:** exibe somente o tamanho aproximado dos principais arrays de entrada e saída; **não** equivale ao pico completo de RAM da execução. Os resultados dependem de executar no Colab.
- **Rótulos e clusters:** IDs de clusters do K-Means são arbitrários (não assumir que `cluster 3` permanece sendo vegetação). A distribuição de classes por cluster é exploratória, pois as referências podem intersectar múltiplos segmentos e não constituem validação independente.
- **Sem novas amostras nem medidas:** os rótulos já existentes são usados apenas para selecionar as janelas e contextualizar a análise. Não é medida de precisão de detecção ou risco elétrico.

**Execução:** depois do pipeline original na V2, execute E5a e E5b. E5b utiliza `clip_poly`, `gdf_full` e `path_img_mult` produzidos pelo notebook. Não é necessário executar E1–E4 para esses dois ensaios. Se a memória for limitada, execute **E5a antes** de E5b e evite manter cópias extras dos rasters.


In [ ]:
# E5a — sensibilidade do QUICKSHIFT em janelas fixas da imagem original
# NÃO substitui segments_quick2 nem altera im/src_rgb ou suas saídas históricas.
import time
import numpy as np
import pandas as pd
from rasterio.transform import rowcol
from skimage.segmentation import quickshift

assert "src_rgb" in globals() and "im" in globals(), "Execute a leitura RGB original."
assert "gdf_arvore" in globals() and "gdf_grama" in globals(), (
    "Execute a leitura das camadas Arvores.gpkg e Grama.gpkg."
)
assert im.ndim == 3 and im.shape[2] >= 3
assert src_rgb.crs is not None
assert (src_rgb.height, src_rgb.width) == im.shape[:2], "Raster e imagem não alinhados."
if min(src_rgb.height, src_rgb.width) < 256:
    raise ValueError("A imagem original deve permitir janelas de 256 x 256 pixels.")

configuracoes_e5_q = [
    (2, 5, 0.04), (3, 3, 0.04), (3, 5, 0.04),
    (3, 8, 0.04), (4, 5, 0.04)
]
lado_e5_q = 256
janelas_e5_q = []
vistos_e5_q = set()

# Dois centros determinísticos por classe, quando houver; base já existente.
for classe_e5_q, camada_e5_q in [("arvore", gdf_arvore), ("grama", gdf_grama)]:
    candidatos_e5_q = camada_e5_q.to_crs(src_rgb.crs)
    candidatos_e5_q = candidatos_e5_q.loc[
        candidatos_e5_q.geometry.notna()
        & ~candidatos_e5_q.geometry.is_empty
    ].copy()
    centroides_e5_q = candidatos_e5_q.geometry.centroid
    locais_e5_q = []
    for centro_e5_q in centroides_e5_q:
        rr_e5_q, cc_e5_q = rowcol(src_rgb.transform, centro_e5_q.x, centro_e5_q.y)
        if 0 <= rr_e5_q < src_rgb.height and 0 <= cc_e5_q < src_rgb.width:
            r0_e5_q = int(np.clip(rr_e5_q - lado_e5_q // 2, 0,
                                  src_rgb.height - lado_e5_q))
            c0_e5_q = int(np.clip(cc_e5_q - lado_e5_q // 2, 0,
                                  src_rgb.width - lado_e5_q))
            if (r0_e5_q, c0_e5_q) not in vistos_e5_q:
                locais_e5_q.append((r0_e5_q, c0_e5_q))
    # Permite manter janelas em diferentes regiões da mesma classe.
    if locais_e5_q:
        escolhas_e5_q = [0, len(locais_e5_q) - 1]
        for ix_e5_q in escolhas_e5_q:
            if len(janelas_e5_q) >= 4:
                break
            coords_e5_q = locais_e5_q[ix_e5_q]
            if coords_e5_q not in vistos_e5_q:
                janelas_e5_q.append((classe_e5_q, *coords_e5_q))
                vistos_e5_q.add(coords_e5_q)

if not janelas_e5_q:
    raise ValueError("Nenhuma janela válida encontrada junto aos rótulos existentes.")
if len(janelas_e5_q) < 4:
    print("Aviso: somente", len(janelas_e5_q),
          "janelas únicas encontradas; as configurações ainda serão comparadas nas mesmas janelas.")

linhas_e5_quick = []
for ind_e5_q, (classe_e5_q, r0_e5_q, c0_e5_q) in enumerate(janelas_e5_q, 1):
    recorte_e5_q = np.ascontiguousarray(
        im[r0_e5_q:r0_e5_q + lado_e5_q, c0_e5_q:c0_e5_q + lado_e5_q, :3]
    )
    for kernel_e5_q, dist_e5_q, ratio_e5_q in configuracoes_e5_q:
        inicio_e5_q = time.perf_counter()
        seg_e5_q = quickshift(
            recorte_e5_q, kernel_size=kernel_e5_q,
            max_dist=dist_e5_q, ratio=ratio_e5_q
        )
        segundos_e5_q = time.perf_counter() - inicio_e5_q
        _, contagens_e5_q = np.unique(seg_e5_q, return_counts=True)
        linhas_e5_quick.append({
            "janela": ind_e5_q, "classe_referencia_centro": classe_e5_q,
            "row0": r0_e5_q, "col0": c0_e5_q,
            "kernel_size": kernel_e5_q, "max_dist": dist_e5_q,
            "ratio": ratio_e5_q, "config_original":
                kernel_e5_q == 3 and dist_e5_q == 5 and ratio_e5_q == 0.04,
            "n_segmentos": int(len(contagens_e5_q)),
            "pixels_medianos_segmento": float(np.median(contagens_e5_q)),
            "fracao_segmentos_menos_10px": float(
                np.mean(contagens_e5_q < 10)
            ),
            "tempo_s": segundos_e5_q,
            "memoria_arrays_mb": (recorte_e5_q.nbytes + seg_e5_q.nbytes) / 2**20,
        })
    print(f"Janela E5a {ind_e5_q}/{len(janelas_e5_q)} concluída.")
tabela_e5_quick = pd.DataFrame(linhas_e5_quick)
print("\n=== E5a — Quickshift, mesmas janelas (256x256) ===")
print(tabela_e5_quick.round(4).to_string(index=False))
print("\nResumo por parâmetros (média entre janelas, não teste independente):")
print(tabela_e5_quick.groupby(
    ["kernel_size", "max_dist", "ratio", "config_original"]
)[["n_segmentos", "pixels_medianos_segmento",
   "fracao_segmentos_menos_10px", "tempo_s"]].agg(["mean", "std"]).round(4))
print("\nNOTA: tempo/memória em janelas pequenas NÃO extrapolam linearmente ao raster inteiro.")


In [ ]:
# E5b — K-Means em uma amostra FIXA de segmentos originais, apenas Area1
# Usa polígonos e bandas multiespectrais existentes; nenhuma segmentação/reclassificação é sobrescrita.
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

assert "clip_poly" in globals() and "gdf_full" in globals(), (
    "Execute as células de clip_poly e gdf_full no pipeline original."
)
assert "path_img_mult" in globals(), "Falta a imagem multiespectral original."
assert clip_poly.crs is not None and gdf_full.crs is not None
if len(clip_poly) < 24:
    raise ValueError("São necessários 24 segmentos originais para K=8.")
# Amostra fixa com segmentos anotados por interseção espacial e parte não anotada.
geometrias_e5_k = clip_poly.loc[
    clip_poly.geometry.notna() & ~clip_poly.geometry.is_empty
].copy()
geometrias_e5_k = geometrias_e5_k.reset_index(drop=True)
referencias_e5_k = gdf_full[["Classe", "geometry"]].to_crs(geometrias_e5_k.crs)
encontro_e5_k = geometrias_e5_k[["geometry"]].sjoin(
    referencias_e5_k, predicate="intersects", how="inner"
)
# Segmentos com rótulos conflitantes ficam fora da análise de classe; NÃO são excluídos do K-Means.
rotulos_e5_k = encontro_e5_k.groupby(level=0)["Classe"].agg(
    lambda x: x.iloc[0] if x.nunique() == 1 else "conflitante"
)
alvos_e5_k = np.array(sorted(rotulos_e5_k.index), dtype=int)
rng_e5_k = np.random.default_rng(42)
max_amostra_e5_k = min(1500, len(geometrias_e5_k))
if len(alvos_e5_k) > max_amostra_e5_k // 2:
    alvos_e5_k = np.sort(rng_e5_k.choice(
        alvos_e5_k, size=max_amostra_e5_k // 2, replace=False
    ))
restantes_e5_k = np.setdiff1d(
    np.arange(len(geometrias_e5_k)), alvos_e5_k, assume_unique=False
)
qtd_outros_e5_k = max_amostra_e5_k - len(alvos_e5_k)
outros_e5_k = rng_e5_k.choice(
    restantes_e5_k, size=qtd_outros_e5_k, replace=False
)
indices_e5_k = np.sort(np.concatenate([alvos_e5_k, outros_e5_k]))
amostra_pol_e5_k = geometrias_e5_k.iloc[indices_e5_k].copy()
lista_e5_k = []
nomes_e5_k = ["R_Mean", "G_Mean", "B_Mean", "N_Mean",
               "R_Std", "G_Std", "B_Std", "N_Std"]

with rasterio.open(path_img_mult) as raster_e5_k:
    if raster_e5_k.count < 4:
        raise ValueError("K-Means E5 requer quatro bandas RGB/NIR no raster original.")
    if amostra_pol_e5_k.crs != raster_e5_k.crs:
        amostra_pol_e5_k = amostra_pol_e5_k.to_crs(raster_e5_k.crs)
    for id_local_e5_k, obj_e5_k in amostra_pol_e5_k.iterrows():
        try:
            imagem_e5_k, _ = mask(
                raster_e5_k, [obj_e5_k.geometry], crop=True, nodata=0
            )
        except (ValueError, rasterio.errors.WindowError):
            continue  # Pode estar fora do raster multiespectral devido à diferença de origem.
        medias_std_e5_k = []
        for banda_e5_k in imagem_e5_k[:4]:
            valores_e5_k = banda_e5_k[banda_e5_k != 0]
            medias_std_e5_k.append(float(np.mean(valores_e5_k)) if valores_e5_k.size else np.nan)
        for banda_e5_k in imagem_e5_k[:4]:
            valores_e5_k = banda_e5_k[banda_e5_k != 0]
            medias_std_e5_k.append(float(np.std(valores_e5_k)) if valores_e5_k.size else np.nan)
        lista_e5_k.append((id_local_e5_k, *medias_std_e5_k))
amostras_e5_k = pd.DataFrame(
    lista_e5_k, columns=["segmento", *nomes_e5_k]
).set_index("segmento")
amostras_e5_k = amostras_e5_k.replace([np.inf, -np.inf], np.nan).dropna()
if len(amostras_e5_k) < 24:
    raise ValueError("Após extrair bandas, restaram menos de 24 segmentos utilizáveis.")
X_e5_k = amostras_e5_k[nomes_e5_k].to_numpy(dtype=float)
metricas_e5_k = []
predicoes_e5_k = {}
for k_e5_k in [4, 6, 8]:
    inicio_e5_k = time.perf_counter()
    modelo_e5_k = KMeans(n_clusters=k_e5_k, random_state=42, n_init=10)
    grupos_e5_k = modelo_e5_k.fit_predict(X_e5_k)
    tempo_e5_k = time.perf_counter() - inicio_e5_k
    predicoes_e5_k[k_e5_k] = grupos_e5_k
    # Silhouette amostral com mesmos itens; não é medida de identificação correta das árvores.
    if 1 < len(np.unique(grupos_e5_k)) < len(X_e5_k):
        sil_e5_k = silhouette_score(
            X_e5_k, grupos_e5_k, sample_size=min(len(X_e5_k), 500),
            random_state=42
        )
    else:
        sil_e5_k = np.nan  # Agrupamento degenerado: silhouette indefinida
    metricas_e5_k.append({
        "k": k_e5_k, "segmentos_validos": len(X_e5_k),
        "inercia": modelo_e5_k.inertia_,
        "silhouette_amostral": sil_e5_k,
        "tempo_s": tempo_e5_k,
        "memoria_matriz_mb": X_e5_k.nbytes / 2**20,
        "ari_vs_k6": adjusted_rand_score(
            predicoes_e5_k[6], grupos_e5_k
        ) if 6 in predicoes_e5_k else np.nan
    })
# K=4 precede K=6; preencher ARI de K=4 em relação ao K=6 após os ajustes.
for resultado_e5_k in metricas_e5_k:
    resultado_e5_k["ari_vs_k6"] = adjusted_rand_score(
        predicoes_e5_k[6], predicoes_e5_k[resultado_e5_k["k"]]
    )
tabela_e5_kmeans = pd.DataFrame(metricas_e5_k)
print("\n=== E5b — sensibilidade K-Means na mesma amostra ===")
print(tabela_e5_kmeans.round(4).to_string(index=False))
for k_e5_k, pred_e5_k in predicoes_e5_k.items():
    print(f"\nK={k_e5_k} — quantidade de segmentos por cluster (IDs sem significado fixo):")
    print(pd.Series(pred_e5_k).value_counts().sort_index().to_string())
    lista_rot_e5_k = rotulos_e5_k.reindex(amostras_e5_k.index).fillna("sem_rotulo")
    tab_e5_k = pd.crosstab(
        pd.Series(pred_e5_k, name="cluster"), lista_rot_e5_k.reset_index(drop=True),
        dropna=False
    )
    print("Cruzamento exploratório com referências já existentes:")
    print(tab_e5_k.to_string())
print("\nATENÇÃO: cluster 3 em K=4, 6 ou 8 NÃO designa a mesma categoria.")
print("Silhouette e ARI medem estrutura/estabilidade do agrupamento,")
print("NÃO precisão da detecção de árvores e NÃO validam a seleção da classe 3.")


## E6 — proximidade a postes: com e sem filtro de classe

**Objetivo:** auditar o resultado histórico da etapa espacial usando **apenas os dados já existentes da Area1**. O fluxo original inclui objetos com `Power_tree_size_C3 > 7` mesmo quando o classificador os marcou como gramínea. Este experimento compara a seleção histórica com a seleção que exige simultaneamente `predicted_classe == 'arvore'` e o mesmo limiar C3. A categoria "grama" é exibida separadamente para ajudar a interpretar a diferença.

**Três cenários:** (1) todos os candidatos C3 > 7, (2) somente os candidatos classificados como árvore, (3) somente os classificados como gramínea. Os cenários utilizam **exatamente os mesmos polígonos, critérios C3 e postes**; somente a condição de classe muda.

**Três distâncias:** 1, 3 e 5 metros. Para evitar inflar as contagens, a E6 usa `sjoin_nearest` em **ambos os sentidos**, uma vez por cenário e com `max_distance=5`; depois filtra a menor distância de cada objeto para cada limiar. Assim, contabiliza tanto **segmentos distintos** quanto **postes distintos** próximos aos segmentos. Não soma as linhas de associações com empates: um poste ou segmento pode aparecer em mais de uma linha, mas é contado apenas uma vez.

**Condições e limitações:** as distâncias são **planimétricas entre polígonos e pontos de postes**, não distâncias aos cabos ou medidas de segurança. `C3 > 7` é apenas uma estimativa empírica de porte, **não altura observada**. O classificador é o **RF histórico**; este teste não demonstra melhoria de acurácia ou risco elétrico real. A E6 não modifica `filter_tree`, `tree_polys`, `Postes`, os 116 blocos originais nem os arquivos de entrada.

**Execução:** execute primeiro as células originais até gerar `tree_polys`, `filter_tree` e `Postes` já reprojetados e recortados para a AOI; depois execute E6a e E6b. **E1–E5 não são pré-requisitos para E6.** Não há resultados novos até executar com os dados reais no Colab.


In [ ]:
# E6a — contagens únicas de segmentos e postes em 1, 3 e 5 m
# Pré-requisitos: execução das células originais até Postes = Postes.clip(AOI).
import numpy as np
import pandas as pd
import geopandas as gpd

for nome_e6 in ["tree_polys", "filter_tree", "Postes"]:
    if nome_e6 not in globals():
        raise RuntimeError(f"Execute as células originais antes da E6: falta {nome_e6}.")
if tree_polys.crs is None or Postes.crs is None:
    raise ValueError("CRS ausente nas geometrias.")
if tree_polys.index.has_duplicates:
    raise ValueError("IDs de segmentos repetidos em tree_polys. Não deduplicar sem auditoria.")
colunas_e6 = ["Power_tree_size_C3", "predicted_classe"]
if any(nome_e6 not in tree_polys.columns for nome_e6 in colunas_e6):
    raise ValueError("Faltam o porte C3 ou as previsões do classificador RF original.")

# CRS projetado em metros, correspondente ao raster original da Area1.
objetos_e6 = tree_polys[["Power_tree_size_C3", "predicted_classe", "geometry"]].copy()
objetos_e6 = objetos_e6.to_crs(epsg=31982)
objetos_e6["segmento_id_e6"] = objetos_e6.index.map(str)
objetos_e6["classe_e6"] = (
    objetos_e6["predicted_classe"].astype(str).str.strip().str.lower()
)
objetos_e6["porte_c3_e6"] = pd.to_numeric(
    objetos_e6["Power_tree_size_C3"], errors="coerce"
)
objetos_e6 = objetos_e6.loc[
    objetos_e6.geometry.notna() & (~objetos_e6.geometry.is_empty) &
    np.isfinite(objetos_e6["porte_c3_e6"])
].copy()
if not objetos_e6.geometry.is_valid.all():
    raise ValueError("Geometrias de candidatos inválidas; reveja os polígonos.")
candidatos_e6 = objetos_e6.loc[objetos_e6["porte_c3_e6"] > 7].copy()
candidatos_e6 = candidatos_e6.reset_index(drop=True)
if len(candidatos_e6) != len(filter_tree):
    print("AVISO: a contagem E6 difere de filter_tree (geometrias vazias/inválidas ou C3 não finito).")

postes_brutos_e6 = Postes[["geometry"]].copy().to_crs(candidatos_e6.crs)
postes_brutos_e6 = postes_brutos_e6.loc[
    postes_brutos_e6.geometry.notna() & (~postes_brutos_e6.geometry.is_empty)
].copy()
if not postes_brutos_e6.geometry.is_valid.all():
    raise ValueError("Geometrias inválidas na camada de postes.")
if not postes_brutos_e6.geometry.geom_type.isin(["Point"]).all():
    raise ValueError("A E6 requer postes representados por pontos.")
# Registros com geometrias idênticas são uma única localização de poste.
postes_brutos_e6["chave_geom_e6"] = postes_brutos_e6.geometry.to_wkb(hex=True)
postes_e6 = postes_brutos_e6.drop_duplicates("chave_geom_e6").copy()
postes_e6 = postes_e6[["geometry"]].reset_index(drop=True)
postes_e6["poste_id_e6"] = np.arange(len(postes_e6), dtype=int)
print("=== E6 | Conferência dos dados ===")
print("Candidatos com C3 > 7:", len(candidatos_e6))
print("Classes previstas entre candidatos:",
      candidatos_e6["classe_e6"].value_counts().to_dict())
print("Postes válidos (antes de retirar coordenadas idênticas):", len(postes_brutos_e6))
print("Postes em localizações distintas:", len(postes_e6))
if not candidatos_e6.empty and not postes_e6.empty:
    assert candidatos_e6["segmento_id_e6"].is_unique
    assert postes_e6["poste_id_e6"].is_unique

cenarios_e6 = {
    "C3 > 7 (histórico)": candidatos_e6,
    "C3 > 7 e árvore": candidatos_e6.loc[candidatos_e6["classe_e6"] == "arvore"],
    "C3 > 7 e gramínea": candidatos_e6.loc[candidatos_e6["classe_e6"] == "grama"],
}
raios_e6 = [1, 3, 5]  # metros, em EPSG:31982
linhas_e6 = []
# Duas direções são necessárias: nearest(polygon -> pole) conta polígonos,
# nearest(pole -> polygon) conta postes. Ambos usam a mesma distância exata.
for cenario_e6, poligonos_e6 in cenarios_e6.items():
    total_candidatos_e6 = len(poligonos_e6)
    if total_candidatos_e6 and len(postes_e6):
        sjoin_seg_e6 = gpd.sjoin_nearest(
            poligonos_e6[["segmento_id_e6", "geometry"]],
            postes_e6[["poste_id_e6", "geometry"]],
            how="inner", max_distance=5, distance_col="dist_m_e6"
        )
        sjoin_post_e6 = gpd.sjoin_nearest(
            postes_e6[["poste_id_e6", "geometry"]],
            poligonos_e6[["segmento_id_e6", "geometry"]],
            how="inner", max_distance=5, distance_col="dist_m_e6"
        )
    else:
        sjoin_seg_e6 = pd.DataFrame(columns=["segmento_id_e6", "dist_m_e6"])
        sjoin_post_e6 = pd.DataFrame(columns=["poste_id_e6", "dist_m_e6"])
    for raio_e6 in raios_e6:
        proximos_seg_e6 = sjoin_seg_e6.loc[
            sjoin_seg_e6["dist_m_e6"] <= raio_e6 + 1e-9
        ]
        proximos_post_e6 = sjoin_post_e6.loc[
            sjoin_post_e6["dist_m_e6"] <= raio_e6 + 1e-9
        ]
        segmentos_proximos_e6 = proximos_seg_e6["segmento_id_e6"].nunique()
        postes_proximos_e6 = proximos_post_e6["poste_id_e6"].nunique()
        linhas_e6.append({
            "cenario": cenario_e6, "distancia_m": raio_e6,
            "segmentos_candidatos": total_candidatos_e6,
            "segmentos_proximos": int(segmentos_proximos_e6),
            "fracao_segmentos_pct": round(
                100 * segmentos_proximos_e6 / total_candidatos_e6, 2
            ) if total_candidatos_e6 else np.nan,
            "postes_total": len(postes_e6),
            "postes_proximos": int(postes_proximos_e6),
            "fracao_postes_pct": round(
                100 * postes_proximos_e6 / len(postes_e6), 2
            ) if len(postes_e6) else np.nan,
        })

tabela_e6 = pd.DataFrame(linhas_e6)
print("\n=== E6 | Contagens únicas por cenário e distância ===")
print(tabela_e6.to_string(index=False))


In [ ]:
# E6b — comparação entre método histórico e método filtrado por classe
# Relatório descritivo, sem inferir validação de altura ou risco de interferência.
import matplotlib.pyplot as plt

if "tabela_e6" not in globals() or tabela_e6.empty:
    raise RuntimeError("Execute E6a antes de E6b.")

historico_e6 = tabela_e6.loc[
    tabela_e6["cenario"] == "C3 > 7 (histórico)"
].set_index("distancia_m")
arvore_e6 = tabela_e6.loc[
    tabela_e6["cenario"] == "C3 > 7 e árvore"
].set_index("distancia_m")
grama_e6 = tabela_e6.loc[
    tabela_e6["cenario"] == "C3 > 7 e gramínea"
].set_index("distancia_m")
comparativo_e6 = pd.DataFrame({
    "distancia_m": historico_e6.index,
    "segmentos_historico": historico_e6["segmentos_proximos"].to_numpy(),
    "segmentos_arvore": arvore_e6["segmentos_proximos"].to_numpy(),
    "segmentos_grama": grama_e6["segmentos_proximos"].to_numpy(),
    "postes_historico": historico_e6["postes_proximos"].to_numpy(),
    "postes_arvore": arvore_e6["postes_proximos"].to_numpy(),
    "postes_grama": grama_e6["postes_proximos"].to_numpy(),
})
comparativo_e6["segmentos_excluidos_pelo_filtro"] = (
    comparativo_e6["segmentos_historico"] - comparativo_e6["segmentos_arvore"]
)
comparativo_e6["postes_deixam_de_ser_sinalizados"] = (
    comparativo_e6["postes_historico"] - comparativo_e6["postes_arvore"]
)
for coluna_e6 in [
    "segmentos_historico", "segmentos_arvore", "segmentos_grama",
    "postes_historico", "postes_arvore", "postes_grama"
]:
    assert (comparativo_e6[coluna_e6].diff().fillna(0) >= 0).all(), (
        f"Não monotonicidade na contagem {coluna_e6}; verificar as associações."
    )
assert (comparativo_e6["segmentos_excluidos_pelo_filtro"] >= 0).all()
assert (comparativo_e6["postes_deixam_de_ser_sinalizados"] >= 0).all()
assert (comparativo_e6["segmentos_historico"] >= comparativo_e6["segmentos_grama"]).all()
# Dentro dos candidatos históricos, as categorias são disjuntas.
# Se existirem outras classes inesperadas, a tabela ainda é válida, mas não somamos como árvore/grama.
print("=== E6 | Comparação do histórico e do filtro de classificação ===")
print(comparativo_e6.to_string(index=False))
print("\nDiferença em segmentos: candidatos próximos que deixam de ser selecionados")
print("quando se exige a previsão de árvore (não são erros comprovados de classificação).")
print("Diferença em postes: localizações próximas que deixam de ser sinalizadas.")
print("Um poste pode estar próximo de segmentos de ambas as classes; contagens de postes")
print("para árvore e gramínea não devem ser somadas para obter o total histórico.")

fig_e6, ax_e6 = plt.subplots(1, 2, figsize=(11, 4))
for nome_e6, eixo_e6 in [
    ("segmentos", ax_e6[0]), ("postes", ax_e6[1])
]:
    eixo_e6.plot(
        comparativo_e6["distancia_m"],
        comparativo_e6[f"{nome_e6}_historico"],
        marker="o", label="C3 > 7 (histórico)"
    )
    eixo_e6.plot(
        comparativo_e6["distancia_m"],
        comparativo_e6[f"{nome_e6}_arvore"],
        marker="s", label="C3 > 7 + árvore"
    )
    eixo_e6.set_xticks([1, 3, 5])
    eixo_e6.set_xlabel("Distância plana ao poste (m)")
    eixo_e6.set_ylabel(f"Número de {nome_e6} distintos")
    eixo_e6.grid(alpha=0.2)
    eixo_e6.legend()
fig_e6.suptitle("E6 — sensibilidade do filtro de classe na proximidade a postes")
fig_e6.tight_layout()
plt.show()
print("\nLIMITAÇÕES: a E6 mede proximidade geométrica a pontos de postes.")
print("Não há geometrias de cabos nem alturas reais que permitam estimar risco elétrico.")


## E6c — diagnóstico de extrapolação do classificador para segmentos sem classe de fundo

O RF histórico foi treinado com **duas classes** (árvore e grama) e aplicado a candidatos escolhidos pelo cluster 3 do K-Means. Não há terceira classe de referência para pavimentos, telhados, água, sombras e demais fundos. Para evitar declarações indevidas, esta etapa mede apenas a **fração dos atributos candidatos dentro/fora da faixa observada no conjunto rotulado**, usando intervalos mínimo–máximo, 1%–99% e 5%–95%. Trata-se de diagnóstico de **suporte marginal de atributos**, não de classificador OOD validado; mesmo segmentos dentro da faixa podem ser falsos positivos.

**Importante:** o cluster 3 é um ID local de um ajuste de K-Means e não representa automaticamente a classe árvore em outros ajustes/recortes. Esta etapa preserva o filtro histórico sem inferir que ele separa toda vegetação do fundo urbano.

In [ ]:
# E6c — auditoria descritiva da extrapolação do conjunto de referência
# Somente df_pred, tree_polys e base_e2 gerados no fluxo histórico da Area1.
import numpy as np
import pandas as pd

for variavel_e6c in ("base_e2", "campos_e2", "df_pred", "tree_polys"):
    if variavel_e6c not in globals():
        raise RuntimeError(f"E6c: execute o fluxo anterior até criar {variavel_e6c}.")
if not all(c in df_pred.columns for c in campos_e2):
    raise ValueError("E6c: df_pred não tem todas as variáveis de treinamento.")
if len(df_pred) != len(tree_polys):
    raise ValueError("E6c: linhas de df_pred e tree_polys desalinhadas.")
referencia_e6c = base_e2[campos_e2].to_numpy(dtype=float)
candidatos_e6c = df_pred[campos_e2].to_numpy(dtype=float)
classes_e6c = tree_polys["predicted_classe"].astype(str).str.strip().str.lower().to_numpy()
if not np.isfinite(referencia_e6c).all():
    raise ValueError("E6c: treinamento contém valores não finitos.")
validos_e6c = np.isfinite(candidatos_e6c).all(axis=1)
tabela_linhas_e6c = []
for q_e6c in (0.0, 0.01, 0.05):
    minimo_e6c = np.quantile(referencia_e6c, q_e6c, axis=0)
    maximo_e6c = np.quantile(referencia_e6c, 1 - q_e6c, axis=0)
    dentro_e6c = validos_e6c & (
        ((candidatos_e6c >= minimo_e6c) & (candidatos_e6c <= maximo_e6c)).all(axis=1)
    )
    arvore_e6c = validos_e6c & (classes_e6c == "arvore")
    tabela_linhas_e6c.append({
        "envelope_referencia": f"{q_e6c*100:.0f}%–{(1-q_e6c)*100:.0f}%",
        "n_candidatos": int(len(df_pred)),
        "n_validos": int(validos_e6c.sum()),
        "n_dentro": int(dentro_e6c.sum()),
        "n_fora_ou_invalidos": int((~dentro_e6c).sum()),
        "pct_fora_ou_invalidos": 100 * float((~dentro_e6c).mean())
            if len(dentro_e6c) else float("nan"),
        "n_previstos_arvore": int(arvore_e6c.sum()),
        "n_arvore_prevista_fora": int((arvore_e6c & ~dentro_e6c).sum()),
    })
tabela_e6c = pd.DataFrame(tabela_linhas_e6c)
print("=== E6c | Suporte marginal dos atributos ===")
print(tabela_e6c.round(2).to_string(index=False))
print("ATENÇÃO: estar dentro de um envelope não valida a classe; estar fora")
print("não prova falso positivo. Sem rótulos de fundo, não há como calcular")
print("precisão operacional da detecção de árvores, distância aos cabos ou risco.")

## E7 — sensibilidade dos modelos alométricos C1–C6

**Objetivo:** verificar como as **12 expressões já usadas na pesquisa** (6 configurações × versões potencial e linear) alteram a seleção de segmentos e as contagens de proximidade a postes. São avaliados os limiares **5, 7 e 9** dos respectivos **índices de porte estimados**. O valor 7 da **C3 potencial** é a referência do código histórico.

**ATENÇÃO CIENTÍFICA:** as equações C1–C6 não foram calibradas com alturas reais observadas na `Area1`. Seus resultados são **índices empíricos, sem acurácia de altura comprovada**. Os limiares 5, 7 e 9 são **cenários numéricos de sensibilidade**, não limites técnicos validados nem alturas medidas em metros. Não calcular MAE, RMSE ou R² de altura sem dados de referência.

Os testes utilizam exclusivamente `tree_polys` e `Postes` do notebook original e **não substituem** `filter_tree`, `tree_polys` ou os resultados anteriores. É feita uma checagem de que as equações recomputadas geram os mesmos valores das colunas originais; se divergirem, o teste é interrompido.

**Planejamento experimental:** para cada expressão e limiar, contar os objetos selecionados **sem filtro de classe** e **com o filtro `predicted_classe == 'arvore'`**; comparar com a configuração histórica C3 > 7 por diferença de contagens e índice de Jaccard; contabilizar os **segmentos distintos** e **postes em localizações distintas** a **1, 3 e 5 m**. A proximidade é computada uma vez entre todos os polígonos elegíveis e uma faixa de 5 m ao redor dos postes existentes, evitando 216 junções espaciais repetidas. Os pares são conferidos pela distância geométrica exata.

**Interpretação:** postes não são cabos de energia; distância 2D ao poste não demonstra risco operacional. Nenhuma variante é considerada cientificamente melhor apenas por selecionar mais ou menos candidatos. Os resultados só estarão disponíveis após rodar E7a e E7b no Colab. E1–E6 não são pré-requisitos; execute primeiro as células originais até `Postes = Postes.clip(AOI)`.


In [ ]:
# E7a — reconstruir C1–C6 (potencial/linear) e preparar proximidade a postes
# Prérequisitos: tree_polys e Postes das células originais.
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

for nome_e7 in ["tree_polys", "Postes"]:
    if nome_e7 not in globals():
        raise RuntimeError(f"Execute o notebook original até criar {nome_e7}.")
if tree_polys.crs is None or Postes.crs is None:
    raise ValueError("tree_polys e Postes precisam ter CRS conhecido.")
if tree_polys.index.has_duplicates:
    raise ValueError("Há índices repetidos em tree_polys. Verifique antes de contabilizar objetos.")
necessarias_e7 = ["area", "predicted_classe"]
if not all(col in tree_polys.columns for col in necessarias_e7):
    raise ValueError("Faltam área do polígono ou classe prevista.")

parametros_e7 = {
    "C1": (0.5, 0.7),
    "C2": (0.6, 0.8),
    "C3": (2.5, 0.4),
    "C4": (4.0, 0.5),
    "C5": (0.4, 0.9),
    "C6": (1.35, 0.72),
}
limiares_e7 = [5.0, 7.0, 9.0]  # valores de índice, NÃO altura observada
raios_e7 = [1.0, 3.0, 5.0]  # metros de distância em planta a POSTES
area_e7 = pd.to_numeric(tree_polys["area"], errors="coerce")
valida_e7 = (
    tree_polys.geometry.notna() & (~tree_polys.geometry.is_empty)
    & area_e7.notna() & np.isfinite(area_e7) & (area_e7 > 0)
)
if (~valida_e7).any():
    print("E7a — objetos descartados por área/geometria inválida:", int((~valida_e7).sum()))
objetos_e7 = tree_polys.loc[valida_e7].copy()
if not objetos_e7.geometry.is_valid.all():
    raise ValueError("Geometrias inválidas; não é seguro calcular distâncias.")
objetos_e7 = objetos_e7.to_crs(epsg=31982)
# Auditar que a área original está em m² — ela foi calculada em tree_polys no fluxo histórico.
area_original_e7 = area_e7.loc[valida_e7].to_numpy(dtype=float)
area_geometrica_e7 = objetos_e7.geometry.area.to_numpy(dtype=float)
if not np.allclose(area_original_e7, area_geometrica_e7, rtol=1e-4, atol=0.05):
    raise ValueError("Área do notebook difere da área geométrica em EPSG:31982; reveja o CRS.")
objetos_e7 = objetos_e7.reset_index(drop=True)
objetos_e7["segmento_id_e7"] = np.arange(len(objetos_e7), dtype=np.int64)
objetos_e7["classe_e7"] = (
    objetos_e7["predicted_classe"].astype(str).str.strip().str.lower()
)
if objetos_e7.empty:
    raise ValueError("Não há objetos com área positiva e geometria válida.")
if "arvore" not in set(objetos_e7["classe_e7"]):
    raise ValueError("Não há objetos previstos como árvore.")
medidas_e7 = {}
for config_e7, (a_e7, b_e7) in parametros_e7.items():
    potencia_e7 = a_e7 * np.power(area_original_e7, b_e7)
    linear_e7 = a_e7 + area_original_e7 * b_e7
    for forma_e7, valores_e7, prefixo_e7 in [
        ("Potencial", potencia_e7, "Power_tree_size_"),
        ("Linear", linear_e7, "Linear_tree_size_")
    ]:
        coluna_original_e7 = prefixo_e7 + config_e7
        if coluna_original_e7 not in objetos_e7.columns:
            raise ValueError(f"Coluna histórica ausente: {coluna_original_e7}")
        registrados_e7 = pd.to_numeric(
            objetos_e7[coluna_original_e7], errors="coerce"
        ).to_numpy(dtype=float)
        if not np.isfinite(registrados_e7).all():
            raise ValueError(f"Valores não finitos em {coluna_original_e7}.")
        if not np.allclose(valores_e7, registrados_e7, rtol=1e-7, atol=1e-7):
            raise AssertionError(f"Expressão e coluna original divergem: {coluna_original_e7}")
        medidas_e7[(config_e7, forma_e7)] = valores_e7

pontos_e7 = Postes[["geometry"]].copy().to_crs(epsg=31982)
pontos_e7 = pontos_e7.loc[
    pontos_e7.geometry.notna() & (~pontos_e7.geometry.is_empty)
].copy()
if not pontos_e7.geometry.is_valid.all():
    raise ValueError("Geometrias inválidas em Postes.")
if not pontos_e7.geometry.geom_type.eq("Point").all():
    raise ValueError("E7 requer geometria pontual dos postes.")
pontos_e7["chave_wkb_e7"] = pontos_e7.geometry.to_wkb(hex=True)
pontos_e7 = pontos_e7.drop_duplicates(subset=["chave_wkb_e7"])
pontos_e7 = pontos_e7[["geometry"]].reset_index(drop=True)
pontos_e7["poste_id_e7"] = np.arange(len(pontos_e7), dtype=np.int64)

# Construção única de pares polígono–poste a até 5 m; efetivamente usados
# os raios 1/3/5 conforme distância geométrica e não somente buffer.
pares_e7 = pd.DataFrame(columns=["segmento_id_e7", "poste_id_e7", "distancia_m_e7"])
if not pontos_e7.empty:
    buffers_e7 = pontos_e7[["poste_id_e7"]].copy()
    buffers_e7 = gpd.GeoDataFrame(
        buffers_e7, geometry=pontos_e7.geometry.buffer(5.000001),
        crs=pontos_e7.crs
    )
    encontros_e7 = gpd.sjoin(
        objetos_e7[["segmento_id_e7", "geometry"]],
        buffers_e7, how="inner", predicate="intersects"
    )
    if not encontros_e7.empty:
        # NumPy com Shapely 2 usa ufunc vetorizada; fallback funciona no Colab
        # com versões anteriores de Shapely.
        ids_obj_e7 = encontros_e7["segmento_id_e7"].to_numpy(dtype=int)
        ids_post_e7 = encontros_e7["poste_id_e7"].to_numpy(dtype=int)
        geoms_obj_e7 = objetos_e7.geometry.to_numpy()[ids_obj_e7]
        geoms_post_e7 = pontos_e7.geometry.to_numpy()[ids_post_e7]
        if hasattr(shapely, "distance"):
            dist_e7 = np.asarray(shapely.distance(geoms_obj_e7, geoms_post_e7))
        else:
            dist_e7 = np.array([
                a.distance(b) for a, b in zip(geoms_obj_e7, geoms_post_e7)
            ])
        pares_e7 = pd.DataFrame({
            "segmento_id_e7": ids_obj_e7,
            "poste_id_e7": ids_post_e7,
            "distancia_m_e7": dist_e7,
        }).loc[lambda x: x["distancia_m_e7"] <= 5.0 + 1e-9]
        pares_e7 = pares_e7.drop_duplicates(["segmento_id_e7", "poste_id_e7"])
print("=== E7a | Dados e conferência de expressões originais ===")
print("Objetos com área válida:", len(objetos_e7))
print("Árvores previstas:", int(objetos_e7["classe_e7"].eq("arvore").sum()))
print("Equações originais conferidas:", len(medidas_e7))
print("Postes em posições únicas:", len(pontos_e7))
print("Pares objeto–poste a até 5 m:", len(pares_e7))
print("Sem alturas observadas ou geometrias de cabos: análise só de sensibilidade.")


In [ ]:
# E7b — matriz de sensibilidade por forma C1–C6, limiar, classe e distância
# Duas seleções: todos os candidatos / somente previsão árvore.
if "medidas_e7" not in globals() or "pares_e7" not in globals():
    raise RuntimeError("Execute E7a antes de E7b.")
base_c3_e7 = medidas_e7[("C3", "Potencial")] > 7
classes_e7 = objetos_e7["classe_e7"].to_numpy()
todos_e7 = np.ones(len(objetos_e7), dtype=bool)
classes_cenario_e7 = {
    "Sem filtro de classe": todos_e7,
    "Somente classe árvore": classes_e7 == "arvore",
}
registros_e7 = []
for filtro_nome_e7, mascara_classe_e7 in classes_cenario_e7.items():
    baseline_e7 = base_c3_e7 & mascara_classe_e7
    conjunto_base_e7 = set(objetos_e7.loc[
        baseline_e7, "segmento_id_e7"
    ].to_numpy(dtype=int))
    for (config_e7, forma_e7), pontuacao_e7 in medidas_e7.items():
        for limiar_e7 in limiares_e7:
            mascara_e7 = (pontuacao_e7 > limiar_e7) & mascara_classe_e7
            ids_e7 = set(objetos_e7.loc[
                mascara_e7, "segmento_id_e7"
            ].to_numpy(dtype=int))
            inter_e7 = len(ids_e7 & conjunto_base_e7)
            uniao_e7 = len(ids_e7 | conjunto_base_e7)
            jaccard_e7 = inter_e7 / uniao_e7 if uniao_e7 else 1.0
            pares_selecionados_e7 = pares_e7.loc[
                pares_e7["segmento_id_e7"].isin(ids_e7)
            ]
            for raio_e7 in raios_e7:
                proximos_e7 = pares_selecionados_e7.loc[
                    pares_selecionados_e7["distancia_m_e7"] <= raio_e7 + 1e-9
                ]
                registros_e7.append({
                    "filtro": filtro_nome_e7, "config": config_e7,
                    "forma": forma_e7, "limiar_indice": limiar_e7,
                    "distancia_poste_m": raio_e7,
                    "n_selecionados": len(ids_e7),
                    "delta_n_vs_c3_7": len(ids_e7) - len(conjunto_base_e7),
                    "jaccard_vs_c3_7": jaccard_e7,
                    "n_segmentos_proximos": int(
                        proximos_e7["segmento_id_e7"].nunique()
                    ),
                    "n_postes_proximos": int(
                        proximos_e7["poste_id_e7"].nunique()
                    ),
                })

tabela_e7 = pd.DataFrame(registros_e7)
esperado_e7 = 2 * 12 * len(limiares_e7) * len(raios_e7)
assert len(tabela_e7) == esperado_e7, "Matriz E7 incompleta."
for _, grupo_e7 in tabela_e7.groupby(["filtro", "config", "forma", "limiar_indice"]):
    assert (grupo_e7.sort_values("distancia_poste_m")[
        ["n_segmentos_proximos", "n_postes_proximos"]
    ].diff().fillna(0).ge(0)).all().all(), "Contagens devem crescer com o raio."

print("=== E7b | Resultado de sensibilidade (12 expressões, 3 limiares) ===")
print("Linhas:", len(tabela_e7), "| comparação histórica: C3 potencial > 7")
print("\nResumo do cenário de índice > 7 (seleção e proximidade até 5 m):")
resumo_e7 = tabela_e7.loc[
    tabela_e7["limiar_indice"].eq(7)
    & tabela_e7["distancia_poste_m"].eq(5)
].sort_values(["filtro", "forma", "config"])
print(resumo_e7.to_string(index=False))
print("\nTabela completa disponível no objeto Python 'tabela_e7' (sem gravar arquivos).")

# Consistência com E6 se a etapa anterior também foi executada na mesma sessão.
if ("tabela_e6" in globals() and isinstance(tabela_e6, pd.Data)
        and not tabela_e6.empty):
    for filtro_e7, nome_e6 in [
        ("Sem filtro de classe", "C3 > 7 (histórico)"),
        ("Somente classe árvore", "C3 > 7 e árvore")
    ]:
        for r_e7 in raios_e7:
            linha_e7 = tabela_e7.loc[
                (tabela_e7["filtro"] == filtro_e7)
                & (tabela_e7["config"] == "C3")
                & (tabela_e7["forma"] == "Potencial")
                & (tabela_e7["limiar_indice"] == 7)
                & (tabela_e7["distancia_poste_m"] == r_e7)
            ].iloc[0]
            linha_e6 = tabela_e6.loc[
                (tabela_e6["cenario"] == nome_e6)
                & (tabela_e6["distancia_m"] == r_e7)
            ]
            if not linha_e6.empty:
                linha_e6 = linha_e6.iloc[0]
                if (linha_e7["n_segmentos_proximos"]
                        != linha_e6["segmentos_proximos"]
                        or linha_e7["n_postes_proximos"]
                        != linha_e6["postes_proximos"]):
                    print(f"ATENÇÃO: divergência E6 vs E7 para {nome_e6} em {r_e7}m."
                          " Verifique tolerâncias e conjunto de objetos elegíveis.")
print("\nIMPORTANTE: mais objetos/postes sinalizados NÃO significa método mais preciso.")


In [ ]:
# E7c — visualizações descritivas, sem atribuir precisão de altura aos modelos
import matplotlib.pyplot as plt

if "tabela_e7" not in globals() or tabela_e7.empty:
    raise RuntimeError("Execute E7b antes de produzir gráficos.")
grafico_e7 = tabela_e7.loc[
    (tabela_e7["filtro"] == "Somente classe árvore")
    & (tabela_e7["limiar_indice"] == 7)
    & (tabela_e7["distancia_poste_m"] == 5)
].copy()
grafico_e7["modelo"] = grafico_e7["config"] + " " + grafico_e7["forma"]
grafico_e7 = grafico_e7.sort_values(["forma", "config"])
plt.figure(figsize=(12, 4))
plt.bar(grafico_e7["modelo"], grafico_e7["n_segmentos_proximos"])
plt.title("E7 — segmentos árvore previstos perto de postes (≤ 5 m, índice > 7)")
plt.ylabel("Quantidade de segmentos distintos")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

sensi_c3_e7 = tabela_e7.loc[
    (tabela_e7["filtro"] == "Somente classe árvore")
    & (tabela_e7["config"] == "C3")
    & (tabela_e7["forma"] == "Potencial")
    & (tabela_e7["distancia_poste_m"] == 5)
].sort_values("limiar_indice")
plt.figure(figsize=(7, 4))
plt.plot(sensi_c3_e7["limiar_indice"],
         sensi_c3_e7["n_segmentos_proximos"], marker="o",
         label="Segmentos próximos")
plt.plot(sensi_c3_e7["limiar_indice"],
         sensi_c3_e7["n_postes_proximos"], marker="s",
         label="Postes próximos")
plt.xticks(limiares_e7)
plt.xlabel("Limiar do índice empírico C3 (não altura observada)")
plt.ylabel("Contagem de elementos distintos próximos (≤ 5 m)")
plt.title("E7 — sensibilidade ao limiar de porte estimado")
plt.legend()
plt.tight_layout()
plt.show()

print("E7 — apenas sensibilidade interna dos modelos C1–C6.")
print("Sem altura real: não calcular MAE, RMSE ou R² nem afirmar acurácia.")
print("Proximidade com postes não comprova interferência com cabos.")


## E8 — consolidação científica e rastreabilidade da execução INTEGRAL da V2

**Importante:** os experimentos DERIVADOS em 223 recortes de 256 × 256 pixels, reproduzidos em 01/11/2026, não equivalem à execução integral deste notebook. A segmentação independente por recorte altera limites/IDs, e a bateria derivada não aplicou o filtro histórico fixo do cluster 3 na classificação E1–E4. Não comparar diretamente seus F1 com a acurácia histórica 0,87.

**Resultados da bateria derivada reproduzida com os dados existentes da Area1:** 510 referências, 505 segmentos rotulados únicos, 10 partições espaciais válidas, 7 subconjuntos de atributos, 216 cenários empíricos de alometria e análise de distância 2D a postes. Esses resultados são apenas exploratórios e não validam o fluxo integral abaixo.

**Ação para completar a validação da V2:** executar as células originais e E1–E7 com o raster inteiro. Depois executar a célula E8: ela salva CSVs, versões das bibliotecas, tamanhos de arquivos de entrada, verificações de consistência e relatório de completude em `Projetos/Segmentacao_Breno/outputs/v2_validacao_integral/`, sem editar os arquivos de entrada ou o README. Se faltarem tabelas, a célula imprime PENDENTE e gera relatório PARCIAL.

**Limites científicos:** validação espacial dentro da mesma AOI não é teste em outra cidade; os parâmetros C1–C6 são índices empíricos de porte sem altura real medida; proximidade em planta a postes não equivale à distância aos cabos ou a risco elétrico confirmado.

In [ ]:
# E8 — consolidação auditável da execução INTEGRAL da V2, sem alterar os dados de entrada.
# Executar após as células originais e E1–E7; não carrega resultados externos/derivados.
from pathlib import Path
from datetime import datetime, timezone
import platform, hashlib, json, importlib.metadata
import pandas as pd

_arquivos_e8 = {
    'E1_auditoria': 'resultado_auditoria_v2',
    'E2_validacao_espacial': 'tabela_e2',
    'E3_classificadores': 'tabela_e3',
    'E3c_robustez_multimodelo': 'tabela_e3c',
    'E4_ablacao': 'tabela_e4',
    'E5a_quickshift': 'tabela_e5_quick',
    'E5b_kmeans': 'tabela_e5_kmeans',
    'E6_postes': 'tabela_e6',
    'E6c_extrapolacao': 'tabela_e6c',
    'E7_alometria': 'tabela_e7',
}
if 'path_img' not in globals():
    raise RuntimeError('Execute primeiro as células originais da V2 (variável path_img ausente).')
_entrada_e8 = Path(path_img)
if not _entrada_e8.is_file():
    raise FileNotFoundError(f'Raster RGB original não encontrado: {_entrada_e8}')
_aoidir_e8 = _entrada_e8.parent
_saida_e8 = _aoidir_e8.parent / 'outputs' / 'v2_validacao_integral'
_saida_e8.mkdir(parents=True, exist_ok=True)

_manifesto_e8 = {
    'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
    'protocolo': 'Segmentacao_area_completa_v2_experimentos.ipynb; E1–E7 após fluxo original',
    'origem': 'Area1: dados existentes; sem novos rótulos, alturas ou geometrias de cabos',
    'python': platform.python_version(),
    'pacotes': {}, 'entradas': {}, 'tabelas': {},
    'avisos': [
        'Métricas da V2 integral não são resultados da bateria DERIVADA em recortes.',
        'Validação espacial interna não demonstra generalização externa.',
        'Modelos C1–C6 não foram calibrados contra alturas observadas.',
        'Distâncias a postes não demonstram interferência com cabos energizados.',
        'Acurácia histórica 0,87 usa outro protocolo e não é diretamente comparável.',
    ],
}
for _pacote_e8 in ['numpy','pandas','geopandas','rasterio','scikit-image','scikit-learn','shapely']:
    try:
        _manifesto_e8['pacotes'][_pacote_e8] = importlib.metadata.version(_pacote_e8)
    except importlib.metadata.PackageNotFoundError:
        _manifesto_e8['pacotes'][_pacote_e8] = 'não encontrado'
for _nome_e8 in ['AOI_RGB_NEW.tif','AOI_NEW_MULTI.tif','AOI_new.shp',
                 'Arvores.gpkg','Grama.gpkg','postes.gpkg']:
    _p_e8 = _aoidir_e8 / _nome_e8
    _manifesto_e8['entradas'][_nome_e8] = {
        'disponivel': _p_e8.is_file(),
        'bytes': _p_e8.stat().st_size if _p_e8.is_file() else None,
    }

for _nome_e8, _variavel_e8 in _arquivos_e8.items():
    _obj_e8 = globals().get(_variavel_e8)
    _ok_e8 = isinstance(_obj_e8, pd.DataFrame) and not _obj_e8.empty
    _registro_e8 = {'variavel': _variavel_e8, 'executada_e_nao_vazia': bool(_ok_e8)}
    if _ok_e8:
        _arquivo_e8 = _saida_e8 / f'{_nome_e8}.csv'
        _obj_e8.to_csv(_arquivo_e8, index=False, encoding='utf-8-sig')
        _registro_e8.update({
            'linhas': int(len(_obj_e8)), 'colunas': list(map(str, _obj_e8.columns)),
            'arquivo': _arquivo_e8.name,
            'sha256': hashlib.sha256(_arquivo_e8.read_bytes()).hexdigest(),
        })
    _manifesto_e8['tabelas'][_nome_e8] = _registro_e8

_completude_e8 = all(t['executada_e_nao_vazia']
                     for t in _manifesto_e8['tabelas'].values())
_manifesto_e8['v2_e1_e7_concluidas_nesta_sessao'] = bool(_completude_e8)

# Testes de consistência sobre os resultados que realmente existirem.
_consistencias_e8 = {}
if isinstance(globals().get('tabela_e3'), pd.DataFrame) and not tabela_e3.empty:
    _n_e3 = tabela_e3.groupby(['bloco_m','rodada'])['modelo'].nunique()
    _consistencias_e8['E3_quatro_modelos_por_particao'] = bool((_n_e3 == 4).all())
if isinstance(globals().get('tabela_e4'), pd.DataFrame) and not tabela_e4.empty:
    _n_e4 = tabela_e4.groupby(['bloco_m','rodada','modelo'])['atributos'].nunique()
    _consistencias_e8['E4_sete_combinacoes_para_modelos_supervisionados'] = bool(
        (_n_e4[_n_e4.index.get_level_values('modelo') != 'Dummy (maioria)'] == 7).all()
    )
if isinstance(globals().get('tabela_e6'), pd.DataFrame) and not tabela_e6.empty:
    _c_e6 = tabela_e6.sort_values('distancia_m')
    _consistencias_e8['E6_contagens_nao_decrescentes_com_raio'] = bool(
        all(grupo['segmentos_proximos'].is_monotonic_increasing and
            grupo['postes_proximos'].is_monotonic_increasing
            for _, grupo in _c_e6.groupby('cenario'))
    )
if isinstance(globals().get('tabela_e7'), pd.DataFrame) and not tabela_e7.empty:
    _consistencias_e8['E7_216_cenarios'] = bool(len(tabela_e7) == 216)
_manifesto_e8['consistencias'] = _consistencias_e8

# Notas de interpretação com resumo descritivo; não são p-valores ou prova de superioridade.
_linhas_relatorio_e8 = [
    '# E8 — consolidação da V2 integral', '',
    f"Gerado: {_manifesto_e8['gerado_em_utc']}",
    f"Execução E1–E7 completa nesta sessão: **{'SIM' if _completude_e8 else 'NÃO'}**", '',
    '## Arquivos e execução',
]
for _etapa_e8, _status_e8 in _manifesto_e8['tabelas'].items():
    _linhas_relatorio_e8.append(f"- {_etapa_e8}: {'gerado' if _status_e8['executada_e_nao_vazia'] else 'não executado/sem linhas'}")
if isinstance(globals().get('tabela_e3'), pd.DataFrame) and not tabela_e3.empty:
    _linhas_relatorio_e8.extend(['', '## E3 — comparação descritiva (F1-macro médio)', ''])
    _linhas_relatorio_e8.append(tabela_e3.groupby(['bloco_m','modelo'])['f1_macro'].agg(['count','mean','std']).round(4).to_string())
if isinstance(globals().get('tabela_e6'), pd.DataFrame) and not tabela_e6.empty:
    _linhas_relatorio_e8.extend(['', '## E6 — contagens espaciais', ''])
    _linhas_relatorio_e8.append(tabela_e6.to_string(index=False))
_linhas_relatorio_e8.extend(['', '## Limitações obrigatórias', *['- '+a for a in _manifesto_e8['avisos']]])
if not _completude_e8:
    _linhas_relatorio_e8.extend(['', '**RELATÓRIO PARCIAL** — não utilizar como comprovação de execução integral da V2.'])
(_saida_e8 / 'relatorio_E8.txt').write_text('\n'.join(_linhas_relatorio_e8), encoding='utf-8')
(_saida_e8 / 'manifesto_E8.json').write_text(
    json.dumps(_manifesto_e8, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'E8 | Pasta de resultados: {_saida_e8}')
for _etapa_e8, _status_e8 in _manifesto_e8['tabelas'].items():
    print(f"  {_etapa_e8}: {'OK' if _status_e8['executada_e_nao_vazia'] else 'PENDENTE'}")
print('E1–E7 completos:', _completude_e8)
print('Checagens:', _consistencias_e8)
if not _completude_e8 or not all(_consistencias_e8.values()):
    print('ATENÇÃO: relatório parcial ou inconsistência; não declarar validação integral concluída.')


## E8 — discussão científica preliminar: evidências da bateria DERIVADA, não da V2 integral

**Origem e limites:** estes números foram reproduzidos em 01/11/2026 com os 223 recortes que continham os 510 rótulos preexistentes de `Area1`. Eles NÃO correspondem à execução integral da V2, pois o Quickshift foi reiniciado por recorte e a classificação não usou a pré-seleção histórica fixa do cluster 3.

| Blocos espaciais | RF (500 árvores) | SVM RBF | Regressão logística | Dummy maioria |
|---:|---:|---:|---:|---:|
| 250 m | 0,8627 | 0,8946 | **0,8981** | 0,2903 |
| 500 m | 0,8499 | **0,8970** | 0,8963 | 0,2899 |

Os valores são F1-macro médio descritivo de cinco partições por escala. As partições reutilizam dados da mesma área; estes resultados não demonstram superioridade estatística ou generalização externa.

### Matriz de argumentos para a revisão bibliográfica

| Questão | Evidência efetivamente disponível | Limite da conclusão |
|---|---|---|
| Qual a contribuição de NDVI, brilho e bordas? | Sete combinações de atributos, avaliadas nas mesmas partições; combinação completa com o maior F1-macro médio para RF, SVM e regressão logística. | É ablação exploratória, não demonstra contribuição causal ou superioridade em outras bases. |
| Por que a abordagem híbrida e orientada a objetos? | Integra segmentação, atributos espectrais/estruturais, classificação e geometria georreferenciada para análise espacial. | A integração é uma característica metodológica; demonstrar sua vantagem demanda protocolo comparativo equivalente. |
| RF é melhor que outras técnicas? | SVM e regressão logística obtiveram F1-macro médio mais alto nesta bateria. | Não sustentar a seleção de RF alegando desempenho superior; apresentar essa observação e discutir critérios de interpretabilidade/integração e limitações. |
| Outros detectores, como YOLO, foram descartados por inferioridade? | Não há experimento comparável com anotações para detecção de objetos no mesmo protocolo. | Não alegar inferioridade de YOLO, deep learning ou LiDAR; explicar restrições do desenho atual. |
| O módulo espacial identifica risco elétrico? | Contagens de objetos/segmentos próximos a pontos de postes em 1, 3 e 5 m. | Não há geometria dos cabos, distâncias 3D ou confirmação de risco operacional. |
| A alometria fornece altura validada? | 12 expressões e 216 cenários de sensibilidade de limiar e raio. | Sem alturas reais, trata-se de índice empírico de porte; não calcular MAE, RMSE ou R² de altura. |

**Interpretação de E8:** as evidências atuais favorecem uma contribuição de *integração metodológica* e análise crítica de sensibilidade, não uma afirmação de superioridade do Random Forest ou previsão confirmada de altura/risco. Na dissertação, confrontar esses argumentos com os textos completos dos estudos citados, explicitando diferenças de dados, tarefa e protocolo; comparações numéricas entre estudos heterogêneos não devem ser tratadas como benchmarks diretos.

**Condição para validação final:** executar o raster completo na V2, gerar as oito tabelas esperadas, conferir o manifesto E8 e discutir somente métricas dessa execução. Este Markdown é um registro separado dos testes derivados já reproduzidos.

## E8 — matriz de literatura e argumentos para a banca (verificação 01/11/2026)

Esta matriz orienta a **discussão da revisão bibliográfica** com base em cinco artigos fornecidos pelo pesquisador. A proposta não é provar que outros métodos “não funcionam”, mas explicar a diferença de **problema, tipo de dado, rótulos, validação e escopo**. Fontes: Gazzea et al. (2022), IEEE Transactions on Power Delivery 37(1); Haroun et al. (2021), Energies 14, 3393, DOI 10.3390/en14123393; Ramli & Tahar (2020), Geo-spatial Information Science 23(3), DOI 10.1080/10095020.2020.1805366; Scarmana & McDougall (2023), ISPRS Archives XLVIII-1/W1-2023; Chatziathanasiou, Kitikidou & Milios (2024), Land 13, 1579, DOI 10.3390/land13101579.

| Estudo | Método/validação descritos no artigo | Diferença em relação à Area1 | Como utilizar na argumentação |
|---|---|---|---|
| **Gazzea et al. (2022)** | Monitoramento semissupervisionado de vegetação com satélite de alta resolução; avaliação relacionada a levantamentos LiDAR de rede de distribuição na Noruega | A base Area1 tem imagem RGB/NIR, referências de vegetação e **postes**, mas não cabos ou levantamento LiDAR de altura | Sustenta relevância do monitoramento espacial, sem afirmar que a Area1 avalia o mesmo risco tridimensional |
| **Haroun et al. (2021)** | SVM e análise de descritores cromáticos/texturais para densidade de vegetação em corredor de transmissão | Problema e rótulos não são os mesmos da classificação árvore/grama por segmentos | Justifica **incluir SVM como baseline**, não comparar diretamente taxas de acerto de tarefas diferentes |
| **Ramli & Tahar (2020)** | Delineamento de copas de palmeiras por Seeded Region Growing a partir de UAV; comparação das alturas estimadas com alturas medidas | A Area1 não contém alturas de referência | Fundamenta por que a alometria C1–C6 aqui é **análise de sensibilidade**, não validação de altura |
| **Scarmana & McDougall (2023)** | Relações entre diâmetro à altura do peito, diâmetro de copa e altura, a partir de medidas de 103 árvores da espécie estudada | Espécie/contexto e ausência de calibração local diferem da Area1 | Evitar transferir coeficientes alométricos sem estudo específico e medidas |
| **Chatziathanasiou et al. (2024)** | Modelos largura de copa–altura para três espécies urbanas; 700 árvores medidas em cidades da Grécia | O índice C3 da V2 relaciona **área projetada** a um valor empírico, não replica coeficientes do artigo | Explicar necessidade de calibração local e distinção entre estimativa experimental e altura real |

### O que os experimentos permitem ou não afirmar

- **E1:** duplicação de segmentos pode inflar avaliações feitas por divisão aleatória por linhas; verificar grupos e vazamento.
- **E2:** validação espacial por blocos da Area1 reduz dependência local, mas não demonstra generalização para cidades distintas.
- **E3:** SVM e regressão logística são comparadores relevantes. Se tiverem F1 maior que Random Forest na mesma partição, isso deve ser relatado, e não ocultado.
- **E4:** ablação verifica se a combinação NDVI + brilho HSV + bordas Canny fornece desempenho descritivo superior às combinações individuais na mesma base.
- **E5:** alterar Quickshift/K-Means muda a identidade dos segmentos; silhouette não é F1 de classificação.
- **E6/E7:** seleção de segmentos por índice de porte e distância **2D a postes** não constitui detecção de risco aos cabos nem validação de altura.
- **E8:** apresentar tabelas com protocolo, tamanho amostral, média/desvio-padrão e limitações; guardar artefatos e versões.

**Hipótese científica a investigar, não conclusão antecipada:** combinar atributos espectrais/radiométricos/estruturais na análise orientada a objetos pode ser útil para distinguir árvores e gramíneas e conservar a geometria dos objetos para priorização espacial junto a postes. O ganho deve ser demonstrado em testes; não alegar superioridade sobre YOLO sem comparações equivalentes ou anotações necessárias.

**Distinção fundamental:** “Execução monolítica original da V2”, “execução por 223 recortes rotulados” e “cobertura completa por blocos 256×256” são protocolos **diferentes** e não devem ter métricas combinadas como se fossem a mesma avaliação.

## E8 — relatório final verificado: cobertura do raster por 812 quadrantes (01/11/2026)

**Protocolo:** processamento **derivado**, com Quickshift original aplicado independentemente em cada quadrante de 256×256 pixels da extensão raster. **Não equivale** à execução do Quickshift uma única vez sobre toda a imagem nem comprova que as células históricas de E1–E7 da V2 terminaram no Colab. Os resultados a seguir foram produzidos por scripts locais reproduzíveis e preservados no pacote de evidências, não são saídas salvas das células desta V2.

**Pacote completo:** [scripts, CSVs, gráficos e relatório no Google Drive](https://drive.google.com/file/d/1kOmeFrmdj4i6x6NewSXhG_tBeUIv67in/view?usp=drivesdk).

### Cobertura e reprodutibilidade

| Indicador | Resultado comprovado |
|---|---:|
| Raster RGB | 7.294 × 6.942 pixels = **50.634.948 pixels** |
| Quadrantes processados da extensão raster | **812/812** |
| Segmentos extraídos em quadrantes independentes | **168.133** |
| Segmentos com oito atributos RGB/NIR finitos | **131.786** |
| Associações ponto–segmento existentes | **510** |
| Segmentos rotulados únicos após deduplicação | **505** |
| Conflitos de rótulo | **0** |
| Sobreposição de IDs numa divisão aleatória por registros | **3** |
| Partições espaciais internas válidas | **10** |
| Cenários de sensibilidade alométrica | **216** |

A ampliação de 223 para 812 quadrantes **não introduziu novos pontos de referência**: os resultados supervisionados E1–E4 são, por isso, iguais aos anteriores. A ampliação atualiza a área coberta por segmentação, a distribuição dos segmentos e as análises não supervisionadas/espaciais. Parte da extensão raster retangular está fora do polígono AOI, fator que também limita a interpretação das contagens totais.

### E3 — F1-macro médio nas partições espaciais

| Bloco | Random Forest (500) | SVM RBF | Regressão logística | Dummy |
|---:|---:|---:|---:|---:|
| 250 m | 0,8627 | 0,8946 | 0,8981 | 0,2903 |
| 500 m | 0,8499 | 0,8970 | 0,8963 | 0,2899 |

**E4:** a combinação NDVI + brilho HSV + bordas Canny apresentou o maior F1-macro médio dentre as sete combinações examinadas para os três classificadores supervisionados, dentro deste protocolo exploratório.

**E5:** K-Means com os **131.786** segmentos de oito atributos válidos apresentou silhouette amostral de **0,3853 (K=4)**, **0,3417 (K=6)** e **0,2939 (K=8)**. Silhouette não avalia a qualidade das classes árvore/gramínea.

### E6 — ALERTA DE GENERALIZAÇÃO/FALSOS POSITIVOS

A etapa espacial identificou **7.594 postes dentro da AOI**. O classificador RF, treinado somente com rótulos de **árvore e gramínea**, foi aplicado a todos os segmentos de uma imagem que também contém fundo urbano, construções e outras superfícies sem classe de referência. Isso gera extrapolação não validada.

**Sinal crítico:** **7.421 de 7.594 postes (97,7%)** ficaram a até 1 m de um segmento **previsto** como árvore e selecionado pelo índice C3. Esse valor é **um indicador de saturação do método**, não uma quantidade de árvores verificadas ou de postes sob risco. Não utilizar essas contagens como resultado operacional. O protocolo derivado **não aplicou o filtro original fixo do cluster 3**; os IDs do K-Means variam entre ajustes e não podem ser transferidos diretamente.

### E7 — alometria e distâncias

Foram calculados **216 cenários de sensibilidade matemática** com C1–C6, duas formas, índices 5/7/9, filtros de classe e distâncias 1/3/5 m a postes. **Não há altura real medida nem geometria dos cabos na base**, portanto não se deve alegar validação de altura, interferência elétrica ou risco operacional.

### Conclusão e limitações para a banca

1. **Cobertura completa por blocos concluída** e comprovada por artefatos reprodutíveis, sem estouro de RAM, utilizando somente os dez arquivos existentes.
2. **Comparações E1–E5 e sensibilidade E6–E7 concluídas neste protocolo derivado**, com as restrições acima. A avaliação da classificação é **interna à Area1** e usa os mesmos rótulos anteriores.
3. **Não concluído:** execução monolítica equivalente do notebook original, generalização externa e validação de classe de fundo não vegetado, altura real ou proximidade aos cabos.
4. **Decisão científica:** antes de apresentar rankings de postes próximos a árvores, é necessário definir e validar uma seleção de candidatos a vegetação com os dados existentes; não é correto transformar o resultado saturado em alegação de risco.

Os arquivos originais de entrada, o notebook original e o `README.md` não foram alterados por esta análise.

## E8 — critérios de encerramento e declaração de evidências (auditoria de 01/11/2026)

Esta seção estabelece o que pode ser declarado **concluído**, **parcial** ou **não validado**. É um controle metodológico e não uma saída de execução. Nenhum resultado numérico abaixo foi recalculado por esta célula.

| Critério | Evidência aceitável | Situação atual |
|---|---|---|
| Integridade dos dados | Apenas dez arquivos da Area1; hashes ou identificação de versões | Arquivos localizados; verificação de hashes integrais não anexada nesta célula |
| Experimentos derivados | Scripts, tabelas e logs da bateria de 812 quadrantes | Documentados e vinculados na seção anterior |
| Execução monolítica do Quickshift | Registro de execução sobre toda a AOI sem reiniciar segmentação por quadrante | **Não comprovada** |
| Execução integral E1–E7 no notebook V2 | Células executadas em ordem, sem erros, tabelas persistidas e manifesto de proveniência | **Não comprovada** no notebook publicado (sem outputs salvos) |
| Classificação RF/SVM/Logística/Dummy | Mesmos rótulos e partições espaciais, F1-macro, suportes e dispersão | Comparação **derivada interna** documentada; não concluir superioridade geral |
| Generalização para fundo urbano | Validação independente da classe de fundo ou protocolo de rejeição verificado | **Não validada**; a base rotula apenas árvore/grama |
| Identificação de árvores próximas de postes | Verificar candidatos e falso positivo antes de qualquer ranqueamento operacional | **Não validada**; saturação no experimento derivado |
| Altura e risco elétrico | Alturas reais e geometria dos cabos como referência | **Não avaliáveis** com a base existente |
| Revisão bibliográfica | Relacionar pelo menos os estudos discutidos a tarefas, dados, métricas e lacunas | Matriz preliminar documentada; redação final ainda requer integração crítica |

### Regras de apresentação à banca

1. Descrever a bateria de **812/812 quadrantes** como **cobertura integral por blocos independentes**, nunca como execução monolítica equivalente.
2. Relatar F1-macro das validações espaciais como desempenho **interno à única AOI**. RF não foi o melhor método nesse protocolo; SVM e regressão logística alcançaram valores médios maiores.
3. **Não** informar os 97,7% de postes próximos a segmentos previstos como árvore como taxa de risco, precisão de detecção ou número de árvores verificadas. É evidência de possível extrapolação/falso positivo.
4. Tratar C1–C6 como **sensibilidade de índices empíricos de porte**; sem alturas reais, não há validação de altura.
5. Distinguir distância 2D a **postes** de distância 3D a **cabos** e de risco operacional.
6. Não alegar que YOLO, LiDAR ou outras técnicas são inferiores quando não foram comparadas nas mesmas condições experimentais.
7. Para encerrar a validação integral, executar o fluxo original e E1–E8 em ambiente com memória suficiente, guardar manifesto, tabelas, logs e saídas; caso contrário manter o status **parcial**.

**Conclusão defensável no estágio atual:** a metodologia oferece uma cadeia reproduzível de segmentação orientada a objetos, classificação comparativa e análise espacial exploratória com os dados existentes. A avaliação derivada sustenta comparações internas e revela limitações importantes, mas ainda não valida estimativas operacionais de conflito com a rede elétrica nem equivalência com a execução monolítica.


## Revisão bibliográfica ampliada — lacunas, comparação justa e contribuição delimitada

### Formulação rigorosa do problema

**Pergunta central A — classificação:** em imagens RGB/NIR de alta resolução de uma única área urbana, quão bem diferentes métodos supervisionados distinguem **segmentos previamente rotulados** como árvores/gramíneas sob separação espacial e embargo geométrico?

**Pergunta central B — análise espacial:** como regras de seleção de segmentos e índices empíricos de porte afetam **contagens exploratórias** de objetos próximos a pontos de postes? Esta pergunta **não** mede conflitos com cabos, nem altura real, nem risco operacional.

**Contribuição que os dados atuais sustentam:** um fluxo integrado e auditável, orientado a objetos, para **comparação de classificadores em validação espacial**, ablação de NDVI/brilho/bordas, sensibilidade de segmentação e avaliação **exploratória** de proximidade a postes. A vantagem do fluxo é integração e transparência metodológica, **não superioridade algorítmica comprovada** nem solução completa de segurança elétrica.

### Estudos e *gaps* documentados (artigos verificáveis)

| Referência | Evidência na literatura | O que falta ou difere na Area1 | Consequência cientificamente defensável |
|---|---|---|---|
| **Haroun et al. (2021)**, *Energies* 14(12):3393, DOI [10.3390/en14123393](https://doi.org/10.3390/en14123393) | Detecta invasão de vegetação em corredor de transmissão com SVM e atributos da imagem de satélite | Tarefa, rotulação e definição de corredor diferentes | SVM deve ser comparado **dentro da Area1**; não transplantar as métricas do artigo |
| **Ramli & Tahar (2020)**, *Geo-spatial Information Science* 23(3), DOI [10.1080/10095020.2020.1805366](https://doi.org/10.1080/10095020.2020.1805366) | Delimita copas de palmeiras com SRG/UAV e compara estimativas com alturas medidas | A Area1 não contém medições de altura | Testes C1–C6 demonstram apenas sensibilidade matemática; não validação biométrica |
| **Scarmana & McDougall (2023)**, *ISPRS Archives*, DOI [10.5194/isprs-archives-XLVIII-1-W1-2023-437-2023](https://doi.org/10.5194/isprs-archives-XLVIII-1-W1-2023-437-2023) | Estuda relações biométricas de árvores em ambiente urbano | Relações dependem de espécie e contexto, sem calibração local da Area1 | Coeficientes da literatura não podem ser assumidos como alturas locais |
| **Chatziathanasiou et al. (2024)**, *Land* 13:1579, DOI [10.3390/land13101579](https://doi.org/10.3390/land13101579) | Ajusta relações largura de copa–altura em espécies urbanas medidas na Grécia | Ausência de alturas e espécies correspondentes | Priorizar sensibilidade e incerteza da alometria, não erro de altura sem verdade-terreno |
| **Roberts et al. (2017)**, *Ecography*, DOI [10.1111/ecog.02881](https://doi.org/10.1111/ecog.02881) | Discute viés de validação aleatória em dados com estrutura espacial | Partições da Area1 são internas a uma só AOI | Separar grupos espaciais, testar embargos e explicitar falta de validação externa |
| **Savva et al. (2025)**, *IEEE Geoscience and Remote Sensing Magazine*, DOI [10.1109/MGRS.2025.3558741](https://doi.org/10.1109/MGRS.2025.3558741) | Revisão sobre sensoriamento remoto, IA, vegetação e linhas de energia; aponta barreiras de dados públicos e fusão 2D/3D | A Area1 dispõe de atributos 2D e pontos de postes, não geometria elétrica 3D | Não alegar detecção de colisão com a rede ou resolução do problema tridimensional |
| **Bahreini & Hammad (2025)**, *Forests* 16:1658, DOI [10.3390/f16111658](https://doi.org/10.3390/f16111658) | Revisão de monitoramento da proximidade entre vegetação e redes elétricas | Não existe referência operacional de contato com cabos na Area1 | Posicionamento como **triagem espacial exploratória**, não avaliação de risco |
| **Faisal et al. (2025)**, *Applied Energy* 385:125507, DOI [10.1016/j.apenergy.2025.125507](https://doi.org/10.1016/j.apenergy.2025.125507) | Revisão de deep learning para inspeção de redes; inclui arquiteturas como YOLO | Sem anotações equivalentes de detecção de objetos/cabos nos mesmos recortes | Não alegar que YOLO falha nem que é inferior; comparar **problemas e exigências de rotulação** |

### Hipóteses testáveis na base atual

- **H1 (classificação):** a escolha do algoritmo modifica o F1-macro sob a mesma separação espacial. Teste: E3/E3c com RF, SVM, logística, Dummy, Extra Trees, KNN, HistGradientBoosting e Naive Bayes. **Não assumir sinal ou significância do efeito**.
- **H2 (informação espectral):** a exclusão de NDVI, intensidade ou bordas altera o desempenho. Teste: E4, sete subconjuntos fixos, mesmos testes; avaliar tamanho e direção do efeito.
- **H3 (robustez espacial):** métricas e suporte efetivo podem variar com tamanho de bloco e embargo. Teste: E3c, 250/500 m e 0/25/50 m; registrar exclusões e vazamento.
- **H4 (estabilidade espacial):** contagens próximas a postes são sensíveis às regras de seleção/índice/raio. Teste: E6/E7; **não interpretar contagens como riscos**.
- **H5 (extrapolação):** candidatos fora das faixas de atributos do conjunto rotulado existem e podem ser quantificados. Teste: E6c. Estar dentro da faixa não prova ser árvore; sem negativos urbanos anotados não se estima taxa de falsos positivos real.

### Como responder à questão dos dados públicos levantada pelo orientador

Foram localizadas revisões que discutem datasets públicos e possíveis alternativas de sensores. **Nenhum dado de terceiros foi incorporado à pesquisa**: por decisão do pesquisador, esta etapa deve trabalhar apenas com os dez arquivos existentes na Area1, preservando o escopo e a comparabilidade. Para futura validação externa, novas bases independentes e rótulos compatíveis seriam necessários, mas não são pressuposto das análises exploratórias E1–E8. Não afirmar que dados públicos inexistem; dizer que sua adequação não foi testada nesta etapa.

### Parágrafo-síntese sugerido para discussão

> A literatura demonstra que a identificação e caracterização de árvores próximas à infraestrutura elétrica pode envolver imagens ópticas, segmentação de copas, modelos supervisionados e levantamentos tridimensionais. Entretanto, os estudos diferem em sensores, regiões, classes, formas de anotação e métricas, o que inviabiliza concluir superioridade a partir da comparação de números publicados em tarefas distintas. Neste trabalho, os dados disponíveis foram utilizados para avaliar uma cadeia interpretável e orientada a objetos, com comparação pareada de classificadores, ablação de atributos e validação espacial interna, além de sensibilidade de índices geométricos e de proximidade a postes. Os resultados não validam altura de árvores, posição de cabos nem risco operacional. A ausência de referência para superfícies urbanas não vegetadas e a extrapolação para objetos não rotulados constituem limitações centrais e explicitamente diagnosticadas.

**Status editorial:** texto bibliográfico e desenho dos testes ampliados inseridos; novas células E3c/E6c devem ser **executadas com Area1** e suas saídas preservadas antes que qualquer resultado novo seja declarado.

## E8 — verificação independente dos CSVs do pacote de 812 quadrantes

Foi feita uma **auditoria independente dos artefatos publicados**, lendo os CSVs, logs e manifesto do pacote [mestrado_v2_cobertura_812_resultados_2026-11-01.zip](https://drive.google.com/file/d/1kOmeFrmdj4i6x6NewSXhG_tBeUIv67in/view?usp=drivesdk), sem reprocesar os rasters nesta auditoria. Resultado: **21/21 hashes SHA-256 conferidos**, **220 registros E3/E4**, **10 partições espaciais documentadas**, **22 avaliações por partição**, **0 duplicatas** por bloco/rodada/modelo/atributos. Estes números são verificações do pacote, **não novos experimentos**.

### Comparação com os três atributos (F1-macro médio)

| Blocos | RF (500) | SVM RBF | Logística | Dummy |
|---:|---:|---:|---:|---:|
| 250 m | 0,8627 | 0,8946 | 0,8981 | 0,2903 |
| 500 m | 0,8499 | 0,8970 | 0,8963 | 0,2899 |

Na comparação **pareada**, SVM − RF apresentou diferença média de **+0,0319** (250 m) e **+0,0470** (500 m), com F1 superior em **5/5** e **4/5** rodadas. Logística − RF apresentou diferença média de **+0,0355** (250 m) e **+0,0464** (500 m), com **5/5** e **4/5** rodadas superiores, respectivamente. Descritivo apenas, pois as divisões não são independentes.

A combinação de **três atributos** teve maior F1 médio em **6/6** grupos de avaliação (2 escalas × 3 modelos supervisionados) na ablação derivada. Esse achado sugere complementaridade informativa **nessa amostra e protocolo**, não superioridade em novas cidades.

### Evidências e limitações que devem acompanhar qualquer publicação

- O resultado 812/812 expressa **quadrantes independentes**, não equivalência com Quickshift monolítico; a matriz espacial cobre também o retângulo externo à AOI.
- O valor **7.421/7.594 postes a até 1 m** de objetos previstos como árvore com índice C3 denota **saturação anômala**, não risco. Uma classificação binária árvore/grama **não possui validação do fundo urbano**.
- A execução do notebook de ponta a ponta continua **sem evidência de saídas salvas**. As novas E3c (oito modelos/embargos) e E6c (diagnóstico de extrapolação) estão **implementadas, não executadas** nesta auditoria.
- Os modelos alométricos C1–C6 não têm validação por altura observada. A proximidade a postes é 2D e não determina distância a cabos.
- O pacote foi auditado quanto a integridade e coerência agregada; **não constitui repetição independente dos algoritmos sobre os dados brutos**.

**Critério de encerramento:** somente após a execução dos novos testes com a mesma Area1, conferência dos resultados e definição explícita do escopo (exploratório versus operacional) a versão científica poderá ser declarada consolidada.